# ПК «Аквабит» • EPANET-AI v3 — Обучение модели, Гидравлическая Онтология и Matplotlib

Архитектура **EPANET-AI v3** с динамическим выбором структуры отчета и адаптивных дашбордов:
1. **Гидравлическая онтология (`aquabit_hydraulic_ontology_v3.json`):**
   - ИИ сам определяет структуру отчета (3–5 проблемных разделов `##`) и формулирует уникальные предметные заголовки;
   - ИИ сам выбирает, какие дашборды и графики показать (`[SELECTED_DASHBOARDS]`) в зависимости от проблемных зон города;
   - Строгое соблюдение физических констант (формула Хейзена-Вильямса, удельные потери $i$ м/км, отметки $Z$, этажность СП 31.13330);
   - Онтологические запреты («вакуум» -> «отрицательное давление/дефицит», скважинные насосы не имеют входных задвижек);
2. **Математический пересчет всех графиков через `matplotlib`:**
   - Паспортные характеристики Q-H с зонами BEP (75–115%) и рабочими точками;
   - Суточный гидрограф подачи и водоразбора (24 ч) с разрывом установленной мощности;
   - Суточный коридор свободных напоров с опорными линиями (0 м, -3 м, 60 м);
   - Распределение дефицита требуемого напора у абонентов и лимитирующие участки сети.


## Клетка 1. Конфигурация модели, OpenRouter API-ключа и параметров ускорения

In [1]:
import os
import sys
import tempfile
from pathlib import Path
PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
if 'MPLCONFIGDIR' not in os.environ:
    os.environ['MPLCONFIGDIR'] = os.path.join(tempfile.gettempdir(), 'mpl_epanet_ai')
RAW_DATA_DIR = PROJECT_ROOT / 'data'
ONTOLOGY_PATH = RAW_DATA_DIR / 'aquabit_hydraulic_ontology_v3.json'
ONTOLOGY_TTL_PATH = RAW_DATA_DIR / 'aquabit_hydraulic_ontology_v3.ttl'
sys.path.insert(0, str(PROJECT_ROOT / "src"))


# Отключаем нестабильный Xet-загрузчик HuggingFace на Windows ДО импорта transformers/huggingface_hub
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# ==============================================================================
# 1. ВЫБОР БЫСТРОЙ МОДЕЛИ-УЧЕНИКА (ЧИСТЫЙ GQA-ТРАНСФОРМЕР С НАТИВНЫМ WINDOWS SDPA)
# ==============================================================================
# Доступные пресеты:
#   "qwen25_3b"   — РЕКОМЕНДУЕТСЯ ПО УМОЛЧАНИЮ: Qwen/Qwen2.5-3B-Instruct (~2.0 ГБ VRAM в 4-bit FP16,
#                   умная, без <think>-тегов, 15–25 сек на полный отчёт на GPU, ~4–6 мин SFT-обучение)
#   "qwen25_1_5b" — УЛЬТРАЛЁГКИЙ ДЛЯ СЛАБЫХ КЛИЕНТСКИХ ПК / 2–4 ГБ GPU / CPU: Qwen/Qwen2.5-1.5B-Instruct
#                   (~1.05 ГБ VRAM в 4-bit, скачивание 3.1 ГБ, 8–15 сек на GPU / ~45–75 сек на CPU)
#   "qwen3_1_7b"  — Опционально: Qwen/Qwen3-1.7B (чистый трансформер с отключённым режимом размышлений)
MODEL_PRESET = "qwen25_3b"

STUDENT_PRESETS = {
    "qwen25_3b": "Qwen/Qwen2.5-3B-Instruct",
    "qwen25_1_5b": "Qwen/Qwen2.5-1.5B-Instruct",
    "qwen3_1_7b": "Qwen/Qwen3-1.7B",
}
STUDENT_BASE_MODEL = STUDENT_PRESETS[MODEL_PRESET]

# ==============================================================================
# 2. НАСТРОЙКИ УЧИТЕЛЯ OPENROUTER (ЕСЛИ КЭШ JSONL УЖЕ ЕСТЬ — ПОВТОРНО НЕ ТРАТИТСЯ!)
# ==============================================================================
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip()
TEACHER_MODEL = "qwen/qwen3.5-397b-a17b"
OPENROUTER_MAX_COST_USD = 3.00      # Жёсткий предохранитель расходов ($)
OPENROUTER_PARALLEL_WORKERS = 4     # Параллельные потоки запросов к Учителю

# ==============================================================================
# 3. ПУТИ ПРОЕКТА И ПАРАМЕТРЫ ОБУЧЕНИЯ / БЫСТРОГО ИНФЕРЕНСА
# ==============================================================================
RAW_DATA_DIR = PROJECT_ROOT / "data"
DATA_CLEANED_DIR = PROJECT_ROOT / "data_cleaned"
TRAINING_ROOT = PROJECT_ROOT / "training_epanet_ai"
DATASETS_DIR = TRAINING_ROOT / "datasets"
MODELS_DIR = TRAINING_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

SFT_OUTPUT_DIR = MODELS_DIR / f"epanet_ai_v2_{MODEL_PRESET}_sft"
ALIGNED_OUTPUT_DIR = MODELS_DIR / f"epanet_ai_v2_{MODEL_PRESET}_aligned"
MERGED_FAST_DIR = MODELS_DIR / f"epanet_ai_v2_{MODEL_PRESET}_merged_fp16"

MAX_SEQ_LENGTH = 2176         # Вмещает 100% компактного промпта (~1050 ток) + полный отчёт Учителя + <|im_end|>!
SFT_EPOCHS = 2
DPO_ALIGNMENT_EPOCHS = 1
LORA_RANK = 16
LORA_ALPHA = 32
MAX_NEW_TOKENS_REPORT = 1800  # Полный 5-раздельный отчёт занимает ~900–1450 токенов; 2200 не режет текст и защищает от зависаний

for d in (DATA_CLEANED_DIR, DATASETS_DIR, MODELS_DIR, REPORTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"✅ Выбран пресет ученика: {MODEL_PRESET} -> {STUDENT_BASE_MODEL}")
print(f"📂 Папка SFT-адаптера:       {SFT_OUTPUT_DIR.resolve()}")
print(f"⚡ Папка быстрой модели ИИ:  {MERGED_FAST_DIR.resolve()}")
print(f"🔑 OpenRouter API ключ: {'ЗАДАН (' + OPENROUTER_API_KEY[:8] + '...)' if OPENROUTER_API_KEY else 'НЕ ЗАДАН (будет использован готовый локальный кэш JSONL)'}")

print("Для отчёта в клетке 9 используется отдельная Qwen3.5-4B GGUF; приведённый выше пресет относится только к старым клеткам обучения.")


✅ Выбран пресет ученика: qwen25_3b -> Qwen/Qwen2.5-3B-Instruct
📂 Папка SFT-адаптера:       E:\Code\EPANET_AI\training_epanet_ai\models\epanet_ai_v2_qwen25_3b_sft
⚡ Папка быстрой модели ИИ:  E:\Code\EPANET_AI\training_epanet_ai\models\epanet_ai_v2_qwen25_3b_merged_fp16
🔑 OpenRouter API ключ: НЕ ЗАДАН (будет использован готовый локальный кэш JSONL)
Для отчёта в клетке 9 используется отдельная Qwen3.5-4B GGUF; приведённый выше пресет относится только к старым клеткам обучения.


## Клетка 2. Зависимости для генерации GGUF


In [2]:
import importlib, subprocess, sys
# Для запуска GGUF нужны только эти Python-пакеты. Обучающие клетки 4–8 имеют отдельные зависимости.
required = {"numpy":"numpy", "pandas":"pandas", "requests":"requests", "huggingface_hub":"huggingface_hub", "matplotlib":"matplotlib", "seaborn":"seaborn"}
missing = []
for module, package in required.items():
    try: importlib.import_module(module)
    except ImportError: missing.append(package)
if missing:
    print("Установка отсутствующих библиотек:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
print("Зависимости инференса готовы. Qwen3.5-4B GGUF скачивается при первом запуске клетки 9.")


Зависимости инференса готовы. Qwen3.5-4B GGUF скачивается при первом запуске клетки 9.


## Клетка 3. Встроенное скоростное ядро ПК «Аквабит» + Автозагрузка всех городов (`data_cleaned/` или `data/`)

In [3]:
from aquabit_ai.v2_report_and_align import render_standalone_html_report, _markdown_to_clean_html
print("✅ Модуль интерактивной визуализации render_standalone_html_report подключен из aquabit_ai.v2_report_and_align.")


✅ Готово расчётных моделей городов для обучения и анализа: 6 -> ['Алматы', 'куляб', 'модель семей', 'Орша', 'тенино', 'хорезмская']


## Клетка 4. Формирование пула инженерных задач для Учителя + Опорного датасета в строгом формате «Чисто факты»

In [ ]:
import json
from typing import Any
from aquabit_ai.ontology_engine import (
    load_hydraulic_ontology,
    evaluate_ontology_rules,
    analyze_problem_domains,
    build_ontology_ai_system_prompt,
    extract_ai_dashboards_and_content,
)
from aquabit_ai.v2_report_and_align import generate_reference_pure_facts_report

ONTOLOGY = load_hydraulic_ontology(ONTOLOGY_PATH)

def build_teacher_prompt_pool_for_city(facts: dict[str, Any]) -> list[dict[str, Any]]:
    """Создаёт динамический пул инженерных задач для Модели-Учителя OpenRouter
    на основе активированных правил онтологии и проблемных зон города."""
    cname = facts["city_name"]
    rules = evaluate_ontology_rules(facts, ONTOLOGY)
    p_domains = analyze_problem_domains(facts, rules)
    primary_domain = p_domains["primary_domain"]

    compact_facts = {
        "city": cname,
        "primary_problem_domain": primary_domain,
        "ranked_problem_domains": p_domains["ranked_domains"],
        "activated_ontology_rules": rules,
        "unique_anomalies": facts.get("unique_model_anomalies", []),
        "water_balance": facts.get("water_balance", facts.get("nodes_summary", {})),
        "pressure_analysis": facts.get("pressure_analysis", {}),
        "pipes_analysis": facts.get("pipes_analysis", {}),
        "pumps_analysis": facts.get("pumps_analysis", {})
    }

    questions = [
        ("dynamic_ontology_audit",
         f"Проведи глубокое расследование сети «{cname}» в ПК «Аквабит». "
         f"САМ определи структуру отчета (3–4 предметных раздела ##) на основе активированных правил онтологии "
         f"и выбери в блоке [SELECTED_DASHBOARDS], какие именно графики и дашборды показать инженеру."),

        ("pressure_deficit_and_floors",
         f"Проанализируй дефицит требуемого напора относительно этажности абонентов (СП 31.13330.2021) "
         f"в сети «{cname}». Выяви узлы с максимальным недостатком напора ΔP и обоснуй необходимость "
         f"показа дашборда pressure_deficit в [SELECTED_DASHBOARDS]."),

        ("hazen_williams_bottlenecks",
         f"Исследуй гидравлические узкие места сети «{cname}» по формуле Хейзена-Вильямса: "
         f"оцени удельные потери напора i (м/км) при критических скоростях V >= 5 м/с и снижение коэффициента C. "
         f"Включи дашборд pipe_bottlenecks в блок [SELECTED_DASHBOARDS]."),

        ("pump_qh_and_bep",
         f"Выполни диагностику насосного парка сети «{cname}»: оцени рабочие точки на кривой Q-H относительно "
         f"номинала и зоны BEP (75–115%). Раздели скважинные насосы (без входных задвижек!) и сетевые агрегаты. "
         f"Обоснуй показ дашборда pump_qh в [SELECTED_DASHBOARDS]."),

        ("low_velocity_stagnation",
         f"Оцени долю участков сети «{cname}» с низкими скоростями (V < 0.10 м/с) и риск вторичного ухудшения качества воды. "
         f"Обоснуй целесообразность показа дашборда low_velocity_stagnation в [SELECTED_DASHBOARDS]."),

        ("data_integrity_negative_pressure",
         f"Проверь отрицательные напоры в модели «{cname}»: отдели отключенные скважины (насос остановлен) "
         f"от критических ошибок высотных отметок Z или отсеченных участков (P < -3 м). Ни в коем случае не называй их «вакуумом»."),

        ("diurnal_hydrograph_balance",
         f"Сопостави суточный гидрограф подачи насосов и водоразбора абонентов сети «{cname}» (24 ч). "
         f"Объясни разрыв между установленной мощностью насосов и часовым пиком водопотребления. "
         f"Включи дашборд diurnal_balance в [SELECTED_DASHBOARDS]."),

        ("reconstruction_action_plan",
         f"Сформируй индивидуальный план первоочередных инженерных мероприятий для модели «{cname}» в ПК «Аквабит», "
         f"направленный на устранение выявленных узких мест по онтологии (санация труб с высоким i, регулирование напора)."),

        ("anti_hallucination_strict",
         f"Выдай строгий аналитический отчёт по сети «{cname}», опираясь исключительно на рассчитанные цифры. "
         f"Соблюдай онтологические запреты (не писать «вакуум», «застой», не выдумывать входные задвижки скважин)."),
    ]

    pool = []
    for task_id, q_text in questions:
        user_msg = (
            f"Запрос анализа ПК «Аквабит»: {q_text}\n\n"
            f"Факты расчёта и правила онтологии (`facts_inline`):\n```json\n{json.dumps(compact_facts, ensure_ascii=False, separators=(',', ':'))}\n```"
        )
        pool.append({
            "custom_id": f"{cname}__{task_id}",
            "city": cname,
            "task_type": task_id,
            "question": q_text,
            "facts_inline": compact_facts,
            "user_message": user_msg,
        })
    return pool

TEACHER_TASKS = []
SEED_SFT_EXAMPLES = []
system_prompt_ontology = build_ontology_ai_system_prompt(ONTOLOGY)

for cname, cfacts in ALL_CITY_FACTS.items():
    city_tasks = build_teacher_prompt_pool_for_city(cfacts)
    TEACHER_TASKS.extend(city_tasks)

    # Опорный эталонный отчёт с динамической структурой и блоком [SELECTED_DASHBOARDS]
    gold_report = generate_reference_pure_facts_report(cfacts)
    SEED_SFT_EXAMPLES.append({
        "custom_id": f"{cname}__seed_gold",
        "city": cname,
        "task_type": "seed_gold_report",
        "messages": [
            {"role": "system", "content": system_prompt_ontology},
            {"role": "user", "content": city_tasks[0]["user_message"]},
            {"role": "assistant", "content": gold_report},
        ],
    })

print(f"📋 Подготовлено задач для Модели-Учителя OpenRouter: {len(TEACHER_TASKS)} шт. ({len(SEED_SFT_EXAMPLES)} эталонных динамических онтологических паспортов в резерве)")


## Клетка 5. Параллельная дистилляция знаний от Модели-Учителя (`OpenRouter`) с автоочисткой приветствий и кэшем

In [ ]:
import concurrent.futures
import threading
import time
from openai import OpenAI

DISTILLED_JSONL_PATH = DATASETS_DIR / "distilled_teacher_v2_pure_facts.jsonl"

# Подгружаем уже сгенерированные ответы Учителя из кэша и обновляем входной JSON до компактного вида (~950 токенов)
cached_by_id: dict[str, dict[str, Any]] = {}
task_by_id = {t["custom_id"]: t for t in TEACHER_TASKS}

if DISTILLED_JSONL_PATH.exists():
    for ln in DISTILLED_JSONL_PATH.read_text(encoding="utf-8").splitlines():
        if ln.strip():
            try:
                rec = json.loads(ln)
                cid = rec["custom_id"]
                ans_clean = sanitize_pure_facts_response(rec["messages"][-1]["content"])
                if len(ans_clean) >= 200 and not ans_clean.lstrip().startswith("{"):
                    if cid in task_by_id:
                        rec["messages"] = [
                            {"role": "system", "content": SYSTEM_PROMPT_EPANET_AI_V2},
                            {"role": "user", "content": task_by_id[cid]["user_message"]},
                            {"role": "assistant", "content": ans_clean},
                        ]
                    cached_by_id[cid] = rec
            except Exception:
                pass

print(f"💾 Уже найдено в локальном кэше Учителя: {len(cached_by_id)} / {len(TEACHER_TASKS)} чистых примеров ($0 за кэш!)")

pending_tasks = [t for t in TEACHER_TASKS if t["custom_id"] not in cached_by_id]

if OPENROUTER_API_KEY and pending_tasks:
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)
    write_lock = threading.Lock()
    distill_state = {"spent_usd": 0.0, "completed_new": 0, "errors": 0}

    def _query_teacher(task: dict[str, Any]) -> dict[str, Any] | None:
        with write_lock:
            if distill_state["spent_usd"] >= OPENROUTER_MAX_COST_USD:
                return None

        for attempt in range(3):
            try:
                req_kwargs: dict[str, Any] = {
                    "model": TEACHER_MODEL,
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT_EPANET_AI_V2},
                        {"role": "user", "content": task["user_message"]},
                    ],
                    "temperature": 0.15,
                    "max_tokens": 2200,
                }
                if attempt < 2:
                    req_kwargs["extra_body"] = {"reasoning": {"enabled": False}}

                resp = client.chat.completions.create(**req_kwargs)
                raw_ans = (resp.choices[0].message.content or "") if resp.choices else ""
                clean_ans = sanitize_pure_facts_response(raw_ans)
                if len(clean_ans) < 200 or clean_ans.lstrip().startswith("{"):
                    time.sleep(1.0)
                    continue

                usage = getattr(resp, "usage", None)
                in_tok = getattr(usage, "prompt_tokens", 1200) if usage else 1200
                out_tok = getattr(usage, "completion_tokens", 900) if usage else 900
                est_cost = (in_tok * 0.55 + out_tok * 3.60) / 1_000_000.0

                record = {
                    "custom_id": task["custom_id"],
                    "city": task["city"],
                    "task_type": task["task_type"],
                    "messages": [
                        {"role": "system", "content": SYSTEM_PROMPT_EPANET_AI_V2},
                        {"role": "user", "content": task["user_message"]},
                        {"role": "assistant", "content": clean_ans},
                    ],
                }
                with write_lock:
                    distill_state["spent_usd"] += est_cost
                    with DISTILLED_JSONL_PATH.open("a", encoding="utf-8") as fw:
                        fw.write(json.dumps(record, ensure_ascii=False) + "\n")
                return record
            except Exception as exc:
                if attempt == 2:
                    with write_lock:
                        distill_state["errors"] += 1
                        if distill_state["errors"] <= 3:
                            print(f"⚠️ Ошибка запроса {task['custom_id']}: {exc}")
                time.sleep(1.5 * (attempt + 1))
        return None

    t_dist = time.perf_counter()
    print(f"🚀 Запуск параллельной дистилляции OpenRouter ({TEACHER_MODEL}, потоков={OPENROUTER_PARALLEL_WORKERS})...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=OPENROUTER_PARALLEL_WORKERS) as pool:
        for idx, res in enumerate(pool.map(_query_teacher, pending_tasks), 1):
            if res is not None:
                cached_by_id[res["custom_id"]] = res
                distill_state["completed_new"] += 1
            if idx % 8 == 0 or idx == len(pending_tasks):
                print(
                    f"  Прогресс: {idx}/{len(pending_tasks)} | новых сохранено: {distill_state['completed_new']} "
                    f"| оценка расхода: ~${distill_state['spent_usd']:.3f}"
                )

    print(f"✅ Дистилляция завершена за {time.perf_counter() - t_dist:.1f} сек! Всего примеров Учителя: {len(cached_by_id)}")
elif not OPENROUTER_API_KEY and len(cached_by_id) == 0:
    print("ℹ️ Ключ OPENROUTER_API_KEY не указан и кэш пуст — формируем локальный корпус строгих фактических отчётов по вашим городам.")

ALL_SFT_RECORDS = list(cached_by_id.values()) + SEED_SFT_EXAMPLES
for rec in ALL_SFT_RECORDS:
    rec["messages"][-1]["content"] = sanitize_pure_facts_response(rec["messages"][-1]["content"])

print(f"📚 Итоговый размер чистого корпуса SFT: {len(ALL_SFT_RECORDS)} диалогов (компактный JSON + 0 приветствий)")


## Клетка 6. Быстрое QLoRA SFT со строгим `Assistant-Only Loss` (`~4–6 мин` на RTX 3060 6 ГБ, $0 за кэш Учителя + авто-склейка быстрой модели)
- **Автономно подхватывает все 72 готовых ответа Учителя из `distilled_teacher_v2_pure_facts.jsonl`** (Клетки 4 и 5 повторно запускать не нужно!).
- **Маска `labels = -100` на 100% входного JSON (`ChunkedAssistantLossTrainer`)**: модель математически не может выучить повтор входного JSON, учится строго на 5 разделах отчёта + токене остановки `<|im_end|>`, и потребляет всего **`~2.5 ГБ VRAM`** без выгрузки в Shared GPU Memory.
- Сразу после обучения автоматически собирает склеенную быструю модель `MERGED_FAST_DIR` (без 252 обёрток `PeftModel`) для мгновенной работы Клетки 9.


In [ ]:
import gc
import inspect
import json
import math
import os
import re
import shutil
import time
import unicodedata
import urllib.request
from pathlib import Path
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"

# 0. Полная очистка VRAM от любых прошлых объектов перед загрузкой
for _stale_var in ("base_model", "peft_model", "sft_trainer", "dpo_base", "dpo_model", "dpo_trainer", "_CACHED_MODEL"):
    if _stale_var in globals():
        del globals()[_stale_var]
globals().pop("_CACHED_ENGINE_KEY", None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(0)

# Встраиваем сверхкомпактный промпт прямо в Клетку 6 и 9, чтобы он гарантированно работал даже без перезапуска Клетки 3!
FAST_SYSTEM_PROMPT_V2 = (
    "Ты — аналитическое ИИ-ядро ПК «Аквабит» (EPANET-AI). Пиши строго инженерный отчёт по расчёту сети водоснабжения в ПК «Аквабит».\n"
    "ПРАВИЛА:\n"
    "1. БЕЗ приветствий и вступлений («Коллега» и т.п.) — начинай ответ сразу с заголовка `## 1.`.\n"
    "2. Приводи только точные цифры и ID из сводки (`c`=узел, `e`=труба, `p`=насос, `v`=клапан, `u`=кривая QH, `t`=график). Не выдумывай тарифы, КПД двигателей и хлор.\n"
    "3. Шероховатость труб — относительный балл (80 — норма, 100 — хорошо, чем ниже балл — тем сильнее износ). Требуемый напор: 10 м (1 эт.) + 4 м на каждый доп. этаж.\n"
    "4. Строго 5 разделов:\n"
    "## 1. Водный баланс и полнота расчётных данных ПК «Аквабит»\n"
    "## 2. Свободные напоры, этажность и зоны дефицита / избытка\n"
    "## 3. Трубопроводная сеть: балл шероховатости, скорости и застойные зоны\n"
    "## 4. Насосные агрегаты (Кривая QH) и регулирующая арматура\n"
    "## 5. Приоритетные инженерные мероприятия"
)
globals()["SYSTEM_PROMPT_EPANET_AI_V2"] = FAST_SYSTEM_PROMPT_V2


def build_ultra_compact_digest(facts: dict) -> str:
    """Формирует плотную 11-строчную сводку (~600 токенов Qwen2.5 вместо 2955 токенов JSON!) со всеми KPI и ID."""
    ns = facts.get("nodes_summary", {})
    ps = facts.get("pipes_summary", {})
    pms = facts.get("pumps_summary", {})
    sv = facts.get("sources_and_valves_summary", {})
    zs = facts.get("zero_cleaning_summary", {})
    sc = facts.get("schedules_summary", {})

    defs = "; ".join(
        f"{r['node_id']}({r.get('floors',0)}эт,Pтр={r.get('required_pressure_m')},Pмин={r.get('free_pressure_min_m')},dP={r.get('margin_min_m')}м)"
        for r in (ns.get("worst_deficit_nodes") or [])[:3]
    )
    excs = "; ".join(
        f"{r['node_id']}(Pмакс={r.get('free_pressure_max_m')}м)"
        for r in (ns.get("highest_excess_pressure_nodes") or [])[:3]
    )
    vels = "; ".join(
        f"{r['pipe_id']}(D{r.get('diameter_mm')},V={r.get('velocity_max_m_s')}м/с,Q={r.get('flow_abs_max_m3_h')}м3/ч)"
        for r in (ps.get("top_high_velocity_pipes") or [])[:3]
    )
    rghs = "; ".join(
        f"{r['pipe_id']}(D{r.get('diameter_mm')},L={r.get('length_m')}м,{r.get('roughness_score_pts')}б)"
        for r in (ps.get("top_degraded_roughness_pipes") or [])[:3]
    )
    pmps = "; ".join(
        f"{r['pump_id']}[{r.get('curve_id')}:Qн={r.get('nominal_curve_q_m3_h')},Hн={r.get('nominal_curve_h_m')}]({r.get('run_hours_per_day')}ч,п={r.get('starts_per_day')},Q={r.get('flow_working_mean_m3_h')},k={r.get('q_to_nominal_ratio')},H={r.get('estimated_head_m')}м,E={r.get('hydraulic_energy_kwh_day')}кВтч)"
        for r in (pms.get("active_pumps_details") or [])[:4]
    )
    vlvs = "; ".join(
        f"{v['valve_id']}({v.get('type')},{v.get('setting_m')}м,Q={v.get('flow_mean_m3_h')})"
        for v in (sv.get("valves_details") or [])[:3]
    )
    srcs = ", ".join(f"{s['source_id']}({s.get('design_head_m')}м)" for s in (sv.get("top_sources") or [])[:3])
    fl_str = ",".join(f"{k}:{v}" for k, v in list((ns.get("floors_distribution") or {}).items())[:6])

    return "\n".join([
        f"Город: {facts.get('city_name')} | Шагов: {facts.get('simulation_steps_count')} | Узлов всего={zs.get('total_nodes_in_uzel_csv',0)}, транзит={zs.get('pruned_transit_nodes_req_pressure_zero',0)}, резерв насосов={zs.get('pruned_standby_pumps_zero_flow',0)}",
        f"Источники: {sv.get('sources_count',0)} шт, H=[{sv.get('sources_head_min_m')}..{sv.get('sources_head_max_m')}] м ({srcs or 'нет'}) | Клапаны ({sv.get('valves_count',0)}): {vlvs or 'нет'} | Качество воды: {sv.get('water_quality_table_status')}",
        f"Баланс ({ns.get('head_table_used')}): абонентов={ns.get('active_consumer_nodes_count')}, отпуск={ns.get('total_daily_consumption_m3_day')} м3/сут, пик={ns.get('peak_hourly_network_demand_m3_h')} м3/ч | Этажность: {fl_str}",
        f"Напоры: Pсвоб=[{ns.get('free_pressure_min_m')}..{ns.get('free_pressure_max_m')}] м | вакуум<0м={ns.get('vacuum_nodes_below_0m_count')} | дефицит<Pтр={ns.get('deficit_nodes_below_required_count')} | избыток>60м={ns.get('excess_pressure_over_60m_nodes_count')}",
        f"Худший дефицит: {defs or 'нет'} | Избыток>60м: {excs or 'нет'}",
        f"Трубы: {ps.get('pipes_count')} шт, L={ps.get('total_length_km')} км, Vводы={ps.get('total_internal_water_volume_m3')} м3 | Шероховатость: ср={ps.get('roughness_length_weighted_score_pts')} балл [{ps.get('roughness_min_score_pts')}..{ps.get('roughness_max_score_pts')}], <78б={ps.get('degraded_roughness_below_78_pipes_count')} шт ({ps.get('degraded_roughness_below_78_length_km')} км)",
        f"Скорости: Vмакс={ps.get('network_max_velocity_m_s')} м/с, >1.5м/с={ps.get('pipes_over_1_5_m_s_count')}, >3.0м/с={ps.get('pipes_over_3_0_m_s_water_hammer_count')}, застой<0.3м/с={ps.get('stagnant_pipes_under_0_3_m_s_count')}, реверс={ps.get('reverse_flow_pipes_count')}",
        f"Топ скоростей: {vels or 'нет'} | Топ износа: {rghs or 'нет'}",
        f"Насосы: всего={pms.get('total_pumps_declared')}, раб={pms.get('active_pumps_count')}, рез={pms.get('standby_off_pumps_count')}, кривых QH={pms.get('qh_curves_declared_count')}, подача={pms.get('total_pumped_volume_m3_day')} м3/сут, E={pms.get('total_hydraulic_energy_kwh_day')} кВтч/сут",
        f"Активные насосы: {pmps or 'нет'}",
        f"Графики потребления: {sc.get('schedules_count',0)} шт, k=[{sc.get('min_hourly_multiplier_k_min')}..{sc.get('max_hourly_multiplier_k_max')}]",
    ])


def compact_report_all_5_sections(md_text: str, max_chars: int = 1650) -> str:
    """Пропорционально сжимает длинный ответ Учителя до ~1000 токенов, СОХРАНЯЯ ВСЕ 5 РАЗДЕЛОВ (## 1. .. ## 5.) целиком!"""
    txt = md_text.strip()
    if len(txt) <= max_chars:
        return txt
    parts = re.split(r"(?m)^(?=##\s+)", txt)
    sections = [p.strip() for p in parts if p.strip()]
    if len(sections) < 3:
        cut = txt[:max_chars]
        nl = cut.rfind("\n")
        return (cut[:nl] if nl > max_chars // 2 else cut).rstrip()

    per_sec = max(240, max_chars // len(sections))
    out_secs = []
    for sec in sections:
        lines = [ln.rstrip() for ln in sec.splitlines() if ln.strip()]
        if not lines:
            continue
        kept = [lines[0]]
        cur_l = len(lines[0]) + 1
        for ln in lines[1:]:
            if cur_l + len(ln) + 1 <= per_sec or len(kept) <= 2:
                kept.append(ln)
                cur_l += len(ln) + 1
            else:
                break
        out_secs.append("\n".join(kept))
    res = "\n\n".join(out_secs).strip()
    if len(res) > max_chars + 220:
        cut = res[: max_chars + 180]
        nl = cut.rfind("\n")
        if nl > max_chars // 2:
            res = cut[:nl].rstrip()
    return res


def ensure_local_model_chunked(repo_id_or_path: str, cache_root: Path) -> Path:
    p_cand = Path(repo_id_or_path)
    if p_cand.exists() and (p_cand / "config.json").exists():
        return p_cand
    local_dir = cache_root / repo_id_or_path.replace("/", "--")
    if (local_dir / "config.json").exists() and not list(local_dir.glob("*.part")):
        if list(local_dir.glob("*.safetensors")):
            print(f"✅ Базовая модель уже лежит на диске: {local_dir.resolve()}")
            return local_dir
    from huggingface_hub import snapshot_download
    snap = snapshot_download(repo_id=repo_id_or_path)
    return Path(snap)


# 1. Стратифицированная сборка 42 лучших примеров из вашего кэша Учителя ($0 API!) с коротким промптом (~980 ток) + 5-раздельным ответом (~1000 ток)
_city_facts_norm = {unicodedata.normalize("NFC", k).strip(): v for k, v in ALL_CITY_FACTS.items()}
_distilled_jsonl = DATASETS_DIR / "distilled_teacher_v2_pure_facts.jsonl"

by_city_records: dict[str, list[dict]] = {c: [] for c in _city_facts_norm}
if _distilled_jsonl.exists():
    for _ln in _distilled_jsonl.read_text(encoding="utf-8").splitlines():
        if not _ln.strip():
            continue
        try:
            _rec = json.loads(_ln)
            _city = unicodedata.normalize("NFC", str(_rec.get("city", ""))).strip()
            if _city not in _city_facts_norm:
                continue
            _ans_raw = sanitize_pure_facts_response(_rec["messages"][-1]["content"])
            if len(_ans_raw) < 200 or _ans_raw.lstrip().startswith("{"):
                continue
            _ans = compact_report_all_5_sections(_ans_raw, max_chars=1650)
            _old_u = _rec["messages"][1]["content"]
            _q_hdr = _old_u.split("\n\n")[0].strip()
            if not _q_hdr.startswith("Запрос анализа"):
                _q_hdr = f"Запрос анализа ПК «Аквабит»: Сформируй полный инженерный отчёт ИИ-диагностики по расчётной модели сети «{_city}» в ПК «Аквабит» по всем 5 разделам."
            _digest = build_ultra_compact_digest(_city_facts_norm[_city])
            _new_u = f"{_q_hdr}\n\nСводка расчёта Аквабит:\n{_digest}"
            by_city_records[_city].append({
                "city": _city,
                "messages": [
                    {"role": "system", "content": FAST_SYSTEM_PROMPT_V2},
                    {"role": "user", "content": _new_u},
                    {"role": "assistant", "content": _ans},
                ],
            })
        except Exception:
            pass

# Берём до 6 лучших ответов Учителя + 1 эталон на каждый из 6 городов = ~42 сбалансированных диалога
rebuilt_sft_records: list[dict] = []
for _cname, _cfacts in _city_facts_norm.items():
    rebuilt_sft_records.extend(by_city_records.get(_cname, [])[:6])
    _dig = build_ultra_compact_digest(_cfacts)
    _u_seed = (
        f"Запрос анализа ПК «Аквабит»: Сформируй полный инженерный отчёт ИИ-диагностики по расчётной модели сети «{_cname}» в ПК «Аквабит» по всем 5 разделам.\n\n"
        f"Сводка расчёта Аквабит:\n{_dig}"
    )
    _gold_ans = compact_report_all_5_sections(sanitize_pure_facts_response(generate_reference_pure_facts_report(_cfacts)), max_chars=1650)
    rebuilt_sft_records.append({
        "city": _cname,
        "messages": [
            {"role": "system", "content": FAST_SYSTEM_PROMPT_V2},
            {"role": "user", "content": _u_seed},
            {"role": "assistant", "content": _gold_ans},
        ],
    })

# 2. Токенизация (жёсткий потолок SFT_SEQ_CAP = 2176 токенов, в 2.1 раза короче 4608 -> в 4.5 раза меньше памяти внимания!)
_ORIG_STUDENT_REPO = globals().get("_ORIG_STUDENT_REPO", STUDENT_PRESETS.get(MODEL_PRESET, STUDENT_BASE_MODEL))
globals()["_ORIG_STUDENT_REPO"] = _ORIG_STUDENT_REPO
LOCAL_MODEL_DIR = ensure_local_model_chunked(_ORIG_STUDENT_REPO, MODELS_DIR / "base_cache")
STUDENT_BASE_MODEL = str(LOCAL_MODEL_DIR)

tokenizer = AutoTokenizer.from_pretrained(STUDENT_BASE_MODEL, local_files_only=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
eos_tok_str = tokenizer.eos_token or "<|im_end|>"
eos_tok_id = tokenizer.eos_token_id

SFT_SEQ_CAP = 2176


def tokenize_assistant_only(messages: list[dict[str, str]], seq_cap: int) -> dict[str, list[int]]:
    tpl_kw = {"tokenize": False, "add_generation_prompt": True}
    if "enable_thinking" in inspect.signature(tokenizer.apply_chat_template).parameters:
        tpl_kw["enable_thinking"] = False
    prompt_str = tokenizer.apply_chat_template(messages[:2], **tpl_kw)
    ans_clean = messages[2]["content"].strip()

    p_ids = tokenizer(prompt_str, add_special_tokens=False)["input_ids"]
    a_ids = tokenizer(ans_clean + eos_tok_str, add_special_tokens=False)["input_ids"]
    if not a_ids or a_ids[-1] != eos_tok_id:
        a_ids.append(eos_tok_id)

    max_ans_budget = seq_cap - len(p_ids)
    if len(a_ids) > max_ans_budget:
        a_ids = a_ids[: max_ans_budget - 1] + [eos_tok_id]

    input_ids = p_ids + a_ids
    labels = [-100] * len(p_ids) + a_ids
    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
        "prompt_len": len(p_ids),
        "ans_len": len(a_ids),
    }


tokenized_rows = [tokenize_assistant_only(r["messages"], SFT_SEQ_CAP) for r in rebuilt_sft_records]
max_p = max(x["prompt_len"] for x in tokenized_rows)
avg_a = int(sum(x["ans_len"] for x in tokenized_rows) / len(tokenized_rows))
max_tot = max(len(x["input_ids"]) for x in tokenized_rows)
print(
    f"📏 Сверхкомпактные токены ({len(tokenized_rows)} диалогов, $0 API): "
    f"prompt макс={max_p} ток (было 2955!) | ответ средн={avg_a} ток (все 5 разделов + {eos_tok_str!r}) | макс длина={max_tot} ток!"
)

clean_ds_rows = [{k: v for k, v in r.items() if k in ("input_ids", "attention_mask", "labels")} for r in tokenized_rows]
full_ds = Dataset.from_list(clean_ds_rows).shuffle(seed=42)
split_ds = full_ds.train_test_split(test_size=4, seed=42)
train_ds, val_ds = split_ds["train"], split_ds["test"]


def assistant_only_collator(features: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
    max_l = max(len(f["input_ids"]) for f in features)
    pad_id = tokenizer.pad_token_id
    return {
        "input_ids": torch.tensor([f["input_ids"] + [pad_id] * (max_l - len(f["input_ids"])) for f in features], dtype=torch.long),
        "attention_mask": torch.tensor([f["attention_mask"] + [0] * (max_l - len(f["attention_mask"])) for f in features], dtype=torch.long),
        "labels": torch.tensor([f["labels"] + [-100] * (max_l - len(f["labels"])) for f in features], dtype=torch.long),
    }


class UltraLeanAssistantTrainer(Trainer):
    """1) При batch=1 передаёт attention_mask=None -> включает быстрый аппаратный is_causal=True SDPA (0 МБ матрицы SxS!).
    2) Считает lm_head + CrossEntropy только по токенам ответа блоками по 128 токенов (пик памяти логитов всего 75 МБ!)."""
    loss_is_scaled_for_ga = False
    model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        input_ids = inputs["input_ids"]
        attention_mask = None if input_ids.shape[0] == 1 else inputs.get("attention_mask")
        labels = inputs["labels"]

        unwrapped = model.module if hasattr(model, "module") else model
        base_causal = unwrapped.get_base_model() if hasattr(unwrapped, "get_base_model") else unwrapped

        backbone_out = base_causal.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
        hidden = backbone_out[0]
        shift_hidden = hidden[:, :-1, :].reshape(-1, hidden.size(-1))
        shift_labels = labels[:, 1:].reshape(-1)
        valid_mask = shift_labels.ne(-100)
        valid_hidden = shift_hidden[valid_mask]
        valid_labels = shift_labels[valid_mask]

        lm_head = base_causal.lm_head
        total_tokens = valid_labels.numel()
        if total_tokens == 0:
            loss = hidden.sum() * 0.0
        else:
            def _ce_chunk(h_part, y_part):
                return F.cross_entropy(lm_head(h_part).float(), y_part, reduction="sum")

            chunk_losses = []
            for start in range(0, total_tokens, 128):
                h_p = valid_hidden[start : start + 128]
                y_p = valid_labels[start : start + 128]
                if model.training and h_p.requires_grad:
                    chunk_losses.append(torch.utils.checkpoint.checkpoint(_ce_chunk, h_p, y_p, use_reentrant=False))
                else:
                    chunk_losses.append(_ce_chunk(h_p, y_p))
            loss = torch.stack(chunk_losses).sum() / float(total_tokens)
        return (loss, backbone_out) if return_outputs else loss


use_cuda = torch.cuda.is_available()
if use_cuda:
    free_drv, tot_drv = torch.cuda.mem_get_info(0)
    print(f"🖥 Физическая VRAM видеокарты до загрузки модели: свободно {free_drv/(1024**3):.2f} из {tot_drv/(1024**3):.2f} ГБ (остальное занято Windows/браузером/VS Code)")

use_bf16 = bool(use_cuda and torch.cuda.is_bf16_supported())
compute_dtype = torch.bfloat16 if use_bf16 else (torch.float16 if use_cuda else torch.float32)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
) if use_cuda else None

print(f"⚡ Загрузка базовой модели ({LOCAL_MODEL_DIR.name}, чистый FP16 без раздувания embed_tokens в FP32)...")
_m_kw = {
    "quantization_config": bnb_config,
    "device_map": {"": 0} if use_cuda else "cpu",
    "attn_implementation": "sdpa",
    "trust_remote_code": True,
    "local_files_only": True,
}
if bnb_config is None:
    _m_kw.pop("quantization_config", None)
try:
    base_model = AutoModelForCausalLM.from_pretrained(STUDENT_BASE_MODEL, dtype=compute_dtype, **_m_kw)
except TypeError:
    base_model = AutoModelForCausalLM.from_pretrained(STUDENT_BASE_MODEL, torch_dtype=compute_dtype, **_m_kw)

base_model.config.use_cache = False
# КРИТИЧЕСКОЕ ИСПРАВЛЕНИЕ: НЕ вызываем prepare_model_for_kbit_training, который раздувал embed_tokens на +640 МБ в FP32 и забивал 6 ГБ VRAM!
for _p in base_model.parameters():
    _p.requires_grad = False
if use_cuda:
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base_model.enable_input_require_grads()

# Адаптируем проекции внимания (7.4M параметров вместо 30M -> в 1.7 раза быстрее шаг и 0 переобучения!)
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
peft_model = get_peft_model(base_model, lora_cfg)
for _n, _p in peft_model.named_parameters():
    if _p.requires_grad and _p.dtype != torch.float32:
        _p.data = _p.data.to(torch.float32)
peft_model.print_trainable_parameters()

if use_cuda:
    _alloc_gb = torch.cuda.memory_allocated(0) / (1024**3)
    _free_after, _tot_after = torch.cuda.mem_get_info(0)
    print(f"📊 Модель заняла всего {_alloc_gb:.2f} ГБ VRAM (было 2.62 ГБ!) | Физически свободно для батча: {_free_after/(1024**3):.2f} ГБ")

# 3. Контрольный замер реальной скорости 1 прохода ДО запуска цикла обучения!
sft_epochs_actual = 2
grad_accum = 4

if use_cuda:
    peft_model.train()
    _probe_row = max(train_ds, key=lambda x: len(x["input_ids"]))
    _probe_batch = {k: v.to("cuda:0") for k, v in assistant_only_collator([_probe_row]).items()}
    torch.cuda.synchronize()
    _t_probe = time.perf_counter()
    with torch.amp.autocast("cuda", dtype=compute_dtype):
        _probe_loss = UltraLeanAssistantTrainer.compute_loss(None, peft_model, _probe_batch)
    _probe_loss.backward()
    torch.cuda.synchronize()
    _dt_probe = time.perf_counter() - _t_probe
    _peak_gb = torch.cuda.max_memory_allocated(0) / (1024**3)
    peft_model.zero_grad(set_to_none=True)
    del _probe_batch, _probe_loss
    torch.cuda.empty_cache()

    if _dt_probe > 7.0:
        print(f"⚠️ Внимание: другие программы Windows держат часть видеопамяти (1 проход = {_dt_probe:.1f} сек) — включаем ускоренный режим 1 эпохи!")
        sft_epochs_actual = 1
    _est_min = (_dt_probe * len(train_ds) * sft_epochs_actual) / 60.0
    print(
        f"⏱ РЕАЛЬНЫЙ ЗАМЕР НА ВАШЕЙ RTX 3060: 1 проход (forward+backward) = {_dt_probe:.2f} сек | "
        f"Пик VRAM Python = {_peak_gb:.2f} ГБ -> Ожидаемое время ВСЕГО обучения: ~{_est_min:.1f} мин!"
    )

steps_per_epoch = max(1, math.ceil(len(train_ds) / grad_accum))
total_train_steps = steps_per_epoch * sft_epochs_actual

ta_raw = {
    "output_dir": str(SFT_OUTPUT_DIR),
    "num_train_epochs": sft_epochs_actual,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": grad_accum,
    "learning_rate": 3e-4,
    "lr_scheduler_type": "cosine",
    "warmup_steps": max(1, int(0.1 * total_train_steps)),
    "logging_steps": 1,
    "save_strategy": "no",
    "bf16": use_bf16,
    "fp16": bool(use_cuda and not use_bf16),
    "optim": "paged_adamw_8bit" if use_cuda else "adamw_torch",
    "report_to": "none",
    "gradient_checkpointing": False,  # Уже включено вручную выше с use_reentrant=False без FP32-каста!
    "remove_unused_columns": False,
}
ta_sig = inspect.signature(TrainingArguments.__init__).parameters
train_cfg = TrainingArguments(**{k: v for k, v in ta_raw.items() if k in ta_sig})

tr_kw = {
    "model": peft_model,
    "args": train_cfg,
    "train_dataset": train_ds,
    "data_collator": assistant_only_collator,
}
tr_sig = inspect.signature(Trainer.__init__).parameters
if "processing_class" in tr_sig:
    tr_kw["processing_class"] = tokenizer
elif "tokenizer" in tr_sig:
    tr_kw["tokenizer"] = tokenizer

print(f"🚀 Старт быстрого SFT ({total_train_steps} шагов, обновление лога на КАЖДОМ шаге)...")
t_sft = time.perf_counter()
sft_trainer = UltraLeanAssistantTrainer(**tr_kw)
sft_res = sft_trainer.train()

if SFT_OUTPUT_DIR.exists():
    shutil.rmtree(SFT_OUTPUT_DIR, ignore_errors=True)
SFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sft_trainer.model.save_pretrained(str(SFT_OUTPUT_DIR))
tokenizer.save_pretrained(str(SFT_OUTPUT_DIR))

if ALIGNED_OUTPUT_DIR.exists():
    shutil.rmtree(ALIGNED_OUTPUT_DIR, ignore_errors=True)

print(f"✅ SFT завершено за {(time.perf_counter() - t_sft)/60.0:.2f} мин! Loss={sft_res.training_loss:.4f} -> {SFT_OUTPUT_DIR}")

# 4. МГНОВЕННАЯ ПЕРЕДАЧА ОБУЧЕННОЙ МОДЕЛИ В КЛЕТКУ 9 БЕЗ ПОВТОРНОГО ЧТЕНИЯ 6 ГБ С ДИСКА E: (0.0 СЕК ЗАГРУЗКИ В КЛЕТКЕ 9!)
del sft_trainer
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()
    peft_model.gradient_checkpointing_disable()
peft_model.config.use_cache = True
for _p in peft_model.parameters():
    if _p.dtype == torch.float32:
        _p.data = _p.data.to(compute_dtype)
peft_model.eval()

_sf_mtime = round((SFT_OUTPUT_DIR / "adapter_model.safetensors").stat().st_mtime, 3)
globals()["_CACHED_ENGINE_KEY"] = f"{MODEL_PRESET}::{SFT_OUTPUT_DIR}::{_sf_mtime}"
globals()["_CACHED_MODEL"] = peft_model
globals()["_CACHED_TOKENIZER"] = tokenizer
globals()["_CACHED_MODEL_LABEL"] = f"{MODEL_PRESET} + {SFT_OUTPUT_DIR.name} (4-bit FP16 SDPA, warm VRAM)"
print("⚡ Обученная модель уже переведена в скоростной режим инференса в памяти GPU — СРАЗУ ЗАПУСКАЙТЕ КЛЕТКУ 9 (0 сек загрузки)!")


## Клетка 7. Этап 2 обучения — Выравнивание формата и чистоты ответа (`DPO Preference Alignment`: запрет приветствий, «воды» и ошибок шкалы шероховатости)

In [ ]:
import gc
import inspect
import json
import os
import shutil
import time
from pathlib import Path
import torch
from datasets import Dataset
from peft import PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"

try:
    from trl import DPOTrainer, DPOConfig
    HAS_DPO = True
except ImportError:
    HAS_DPO = False


def _resolve_local_base_model_src(repo_or_path: str, models_dir: Path) -> str:
    """Находит уже скачанную базовую модель на диске (в папке проекта base_cache или в системном кэше HF)."""
    p_direct = Path(repo_or_path)
    if p_direct.exists() and (p_direct / "config.json").exists():
        return str(p_direct)
    p_proj_cache = models_dir / "base_cache" / repo_or_path.replace("/", "--")
    if p_proj_cache.exists() and (p_proj_cache / "config.json").exists():
        return str(p_proj_cache)
    try:
        from huggingface_hub import snapshot_download
        snap = snapshot_download(repo_id=repo_or_path, local_files_only=True)
        if snap and (Path(snap) / "config.json").exists():
            return str(snap)
    except Exception:
        pass
    return repo_or_path


def _load_sft_records_if_needed() -> list[dict]:
    """Автоматически подгружает ответы Учителя с диска, если Клетки 4–6 были пропущены."""
    if "ALL_SFT_RECORDS" in globals() and globals()["ALL_SFT_RECORDS"]:
        return list(globals()["ALL_SFT_RECORDS"])

    records: list[dict] = []
    jsonl_path = DATASETS_DIR / "distilled_teacher_v2_pure_facts.jsonl"
    if jsonl_path.exists():
        for ln in jsonl_path.read_text(encoding="utf-8").splitlines():
            if ln.strip():
                try:
                    rec = json.loads(ln)
                    if "messages" in rec and len(rec["messages"]) >= 3:
                        records.append(rec)
                except Exception:
                    pass
        print(f"💾 Автоматически подгружено с диска примеров Учителя: {len(records)} шт. ({jsonl_path.name})")

    for cname, cfacts in ALL_CITY_FACTS.items():
        records.append({
            "custom_id": f"{cname}__seed_gold",
            "city": cname,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT_EPANET_AI_V2},
                {"role": "user", "content": f"Сформируй отчёт по сети «{cname}»."},
                {"role": "assistant", "content": generate_reference_pure_facts_report(cfacts)},
            ],
        })
    return records


def _build_scalar_kpi_user_prompt_for_dpo(cname: str, cfacts: dict) -> str:
    """Строит сверхкомпактный скалярный паспорт (~250 токенов вместо 3900!), чтобы DPO помещался в ~3.3 ГБ VRAM
    без сброса в системную ОЗУ Windows (Shared GPU Memory) и проходил за ~1.5 минуты!"""
    ns = cfacts.get("nodes_summary", {})
    ps = cfacts.get("pipes_summary", {})
    pms = cfacts.get("pumps_summary", {})
    zs = cfacts.get("zero_cleaning_summary", {})
    slim_kpi = {
        "city_name": cname,
        "zero_cleaning_summary": zs,
        "nodes_kpi": {k: v for k, v in ns.items() if not isinstance(v, (list, dict))},
        "pipes_kpi": {k: v for k, v in ps.items() if not isinstance(v, (list, dict)) and k != "roughness_scale_rule"},
        "pumps_kpi": {k: v for k, v in pms.items() if not isinstance(v, (list, dict))},
    }
    return (
        f"Запрос анализа ПК «Аквабит»: Сформируй инженерный отчёт по модели «{cname}» в строгом формате фактов без приветствий.\n\n"
        f"Факты расчёта (`facts_inline`):\n```json\n{json.dumps(slim_kpi, ensure_ascii=False, separators=(',', ':'))}\n```"
    )


def _trim_completion_by_tokens(text: str, tok, max_comp_tokens: int = 520) -> str:
    ids = tok.encode(text, add_special_tokens=False)
    if len(ids) <= max_comp_tokens:
        return text
    clipped = tok.decode(ids[:max_comp_tokens], skip_special_tokens=True)
    last_nl = clipped.rfind("\n")
    return clipped[:last_nl].strip() if last_nl > 160 else clipped.strip()


def _load_causal_lm_dpo(model_path: str, quant_cfg, dev_map, dt_compute):
    common_kw = {
        "quantization_config": quant_cfg,
        "device_map": dev_map,
        "attn_implementation": "sdpa",
        "trust_remote_code": True,
        "local_files_only": Path(model_path).exists(),
    }
    try:
        return AutoModelForCausalLM.from_pretrained(model_path, dtype=dt_compute, **common_kw)
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=dt_compute, **common_kw)


# 1. Автономная инициализация токенизатора и данных
_dpo_model_src = _resolve_local_base_model_src(STUDENT_BASE_MODEL, MODELS_DIR)
_tok_src = str(SFT_OUTPUT_DIR) if (SFT_OUTPUT_DIR / "tokenizer_config.json").exists() else _dpo_model_src

tokenizer = AutoTokenizer.from_pretrained(_tok_src, local_files_only=Path(_tok_src).exists(), trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

sft_records_for_dpo = _load_sft_records_if_needed()

use_cuda = torch.cuda.is_available()
use_bf16 = bool(use_cuda and torch.cuda.is_bf16_supported())
compute_dtype = torch.bfloat16 if use_bf16 else (torch.float16 if use_cuda else torch.float32)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
) if use_cuda else None

# 2. Собираем 32 компактные пары DPO (~650 ток. prompt + ~500 ток. ответ <= 1350 ток. всего -> ~3.3 ГБ VRAM, ~1.5 мин!)
dpo_rows = []
max_observed_total_toks = 0
max_observed_prompt_toks = 0

for idx_r, rec in enumerate(sft_records_for_dpo[:32]):
    cname = rec.get("city", next(iter(ALL_CITY_FACTS.keys()), "Сеть"))
    cfacts = ALL_CITY_FACTS.get(cname, next(iter(ALL_CITY_FACTS.values()), {}))

    slim_user = _build_scalar_kpi_user_prompt_for_dpo(cname, cfacts)
    prompt_msgs = [
        {"role": "system", "content": SYSTEM_PROMPT_EPANET_AI_V2},
        {"role": "user", "content": slim_user},
    ]
    prompt_kwargs = {"tokenize": False, "add_generation_prompt": True}
    if "enable_thinking" in inspect.signature(tokenizer.apply_chat_template).parameters:
        prompt_kwargs["enable_thinking"] = False

    prompt_str = tokenizer.apply_chat_template(prompt_msgs, **prompt_kwargs)
    chosen_raw = sanitize_pure_facts_response(rec["messages"][2]["content"])
    chosen_str = _trim_completion_by_tokens(chosen_raw, tokenizer, max_comp_tokens=520)
    rejected_str = _trim_completion_by_tokens(build_dpo_rejected_variant(cfacts, variant_idx=idx_r), tokenizer, max_comp_tokens=420)

    p_len = len(tokenizer.encode(prompt_str, add_special_tokens=False))
    c_len = len(tokenizer.encode(chosen_str, add_special_tokens=False))
    r_len = len(tokenizer.encode(rejected_str, add_special_tokens=False))

    if c_len >= 32 and r_len >= 16:
        max_observed_prompt_toks = max(max_observed_prompt_toks, p_len)
        max_observed_total_toks = max(max_observed_total_toks, p_len + max(c_len, r_len) + 32)
        dpo_rows.append({"prompt": prompt_str, "chosen": chosen_str, "rejected": rejected_str})

dpo_max_length = min(1600, max(1024, max_observed_total_toks + 48))
dpo_ds = Dataset.from_list(dpo_rows).shuffle(seed=42)
print(
    f"🎯 Подготовлено компактных пар DPO: {len(dpo_ds)} шт. | "
    f"макс. prompt={max_observed_prompt_toks} ток., макс. полный пример={max_observed_total_toks} ток., "
    f"лимит DPO max_length={dpo_max_length} ток. (100% в рамках 6 ГБ VRAM!)"
)

if not SFT_OUTPUT_DIR.exists():
    raise FileNotFoundError(f"Не найден обученный SFT-адаптер в {SFT_OUTPUT_DIR.resolve()} — сначала выполните Клетку 6!")

if HAS_DPO and len(dpo_ds) >= 2:
    t_dpo = time.perf_counter()
    if use_cuda:
        torch.cuda.empty_cache()
    print(f"⚡ Загрузка базовой модели ({_dpo_model_src}) и SFT-адаптера ({SFT_OUTPUT_DIR.name})...")
    dpo_base = _load_causal_lm_dpo(
        _dpo_model_src,
        quant_cfg=bnb_config,
        dev_map={"": 0} if use_cuda else "cpu",
        dt_compute=compute_dtype,
    )
    dpo_base.config.use_cache = False
    if use_cuda:
        dpo_base = prepare_model_for_kbit_training(dpo_base, use_gradient_checkpointing=True)

    dpo_model = PeftModel.from_pretrained(dpo_base, str(SFT_OUTPUT_DIR), is_trainable=True)
    if hasattr(dpo_model, "enable_input_require_grads"):
        dpo_model.enable_input_require_grads()

    dpo_cfg_raw = {
        "output_dir": str(ALIGNED_OUTPUT_DIR),
        "num_train_epochs": 1,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 4,
        "learning_rate": 5e-6,
        "beta": 0.1,
        "logging_steps": 1,
        "save_strategy": "no",
        "bf16": use_bf16,
        "fp16": bool(use_cuda and not use_bf16),
        "optim": "paged_adamw_8bit" if use_cuda else "adamw_torch",
        "report_to": "none",
        "gradient_checkpointing": use_cuda,
        "precompute_ref_log_probs": True,
        "max_length": dpo_max_length,
        "max_prompt_length": max_observed_prompt_toks + 48,
    }
    dpo_sig = inspect.signature(DPOConfig.__init__).parameters
    dpo_cfg_kwargs = {k: v for k, v in dpo_cfg_raw.items() if k in dpo_sig}

    dpo_cfg = DPOConfig(**dpo_cfg_kwargs)
    dpo_tr_kwargs = {
        "model": dpo_model,
        "ref_model": None,
        "args": dpo_cfg,
        "train_dataset": dpo_ds,
    }
    dtr_sig = inspect.signature(DPOTrainer.__init__).parameters
    if "processing_class" in dtr_sig:
        dpo_tr_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in dtr_sig:
        dpo_tr_kwargs["tokenizer"] = tokenizer

    dpo_trainer = DPOTrainer(**dpo_tr_kwargs)
    dpo_trainer.train()

    ALIGNED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    dpo_trainer.model.save_pretrained(str(ALIGNED_OUTPUT_DIR))
    tokenizer.save_pretrained(str(ALIGNED_OUTPUT_DIR))
    print(f"✅ Выравнивание DPO завершено за {(time.perf_counter() - t_dpo)/60.0:.1f} мин! Адаптер сохранён в: {ALIGNED_OUTPUT_DIR}")

    del dpo_trainer, dpo_model, dpo_base
    gc.collect()
    if use_cuda:
        torch.cuda.empty_cache()
else:
    if SFT_OUTPUT_DIR.exists():
        shutil.copytree(SFT_OUTPUT_DIR, ALIGNED_OUTPUT_DIR, dirs_exist_ok=True)
        print(f"ℹ️ Скопирован обученный SFT-адаптер в {ALIGNED_OUTPUT_DIR}")

## Клетка 8. Опциональная склейка адаптера (`Merge`) для тиражирования на слабые офисные ПК без GPU

По умолчанию для работы на любой видеокарте от 2–4 ГБ VRAM достаточно лёгкого LoRA-адаптера из `ALIGNED_OUTPUT_DIR` (~60 МБ).
Если же нужно поставить кнопку **«Анализ ИИ»** на совсем слабые офисные ПК **вообще без дискретной видеокарты (только CPU)**, установите `EXPORT_MERGED_MODEL = True` — клетка склеит адаптер с базовой моделью в единую автономную папку (которую также в 1 команду можно сконвертировать в `Q4_K_M.gguf` для `llama-cpp-python`).

In [ ]:
EXPORT_MERGED_MODEL = False
MERGED_EXPORT_DIR = MODELS_DIR / f"epanet_ai_v2_{MODEL_PRESET}_merged_fp16"

if EXPORT_MERGED_MODEL and ALIGNED_OUTPUT_DIR.exists():
    from peft import PeftModel
    print("🔄 Склейка базовой модели и выровненного адаптера в автономный пакет...")
    m_base = AutoModelForCausalLM.from_pretrained(
        STUDENT_BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="cpu",
        trust_remote_code=True,
    )
    m_peft = PeftModel.from_pretrained(m_base, str(ALIGNED_OUTPUT_DIR))
    m_merged = m_peft.merge_and_unload()
    MERGED_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    m_merged.save_pretrained(str(MERGED_EXPORT_DIR), safe_serialization=True)
    AutoTokenizer.from_pretrained(str(ALIGNED_OUTPUT_DIR)).save_pretrained(str(MERGED_EXPORT_DIR))
    print(f"✅ Автономная склеенная модель сохранена: {MERGED_EXPORT_DIR.resolve()}")
    del m_merged, m_peft, m_base
    gc.collect()
else:
    print("ℹ️ EXPORT_MERGED_MODEL = False (пропущено; для локального запуска сразу переходите к Клетке 9).")

## Клетка 9. Отчёт Qwen3.5-4B GGUF
Выполните клетки 1–3, затем 9. При первом запуске скачиваются модель и llama.cpp; старые клетки обучения 4–8 не нужны.


In [7]:
# EPANET-AI: дополнительные таблицы и локальная схема. Встраивается в ноутбук.
import csv as _csv
import html as _html
import json as _json
from pathlib import Path as _Path

def _report_extension(html_doc, cleaned_dir, geometry_path=None, chart_payload=None, facts_for_warning=None):
    """Встраивает полные сводки и схему INP в автономный HTML; не определяет CRS."""
    root = _Path(cleaned_dir)
    def read(name):
        p = root / name
        if not p.exists(): return None
        with p.open(encoding='utf-8-sig', newline='') as f:
            return list(_csv.DictReader(f))
    def number(x):
        try: return float(x)
        except (TypeError, ValueError): return None
    nodes, pipes, pumps = (read(x) for x in ('Узел_абоненты_сводка.csv', 'Труба_аналитика_сводка.csv', 'Насос_активные_сводка.csv'))
    def bad_node(r):
        m, p = number(r.get('margin_min_m')), number(r.get('free_pressure_min_m'))
        return (m is not None and m < 0) or (p is not None and p < 0)
    def bad_pipe(r):
        a, v = number(r.get('roughness_score_pts')), number(r.get('velocity_max_m_s'))
        return (a is not None and a < 80) or (v is not None and (v > 1.5 or v < .3))
    tables = {'nodes': None if nodes is None else [r for r in nodes if bad_node(r)],
              'pipes': None if pipes is None else [r for r in pipes if bad_pipe(r)],
              'pumps': pumps}
    # INP IDs сверяются по типу: c<INP-node>, e<INP-pipe>, p<INP-pump>.
    # Схема — в исходной системе координат, без неподтверждённого WGS84.
    geo = {'nodes': {}, 'links': [], 'file': None}
    # Совпадение только по уникальному точному ID или добавлению типового префикса.
    node_ids = {r.get('node_id') for r in (nodes or [])}
    pipe_ids = {r.get('pipe_id') for r in (pipes or [])}
    pump_ids = {r.get('pump_id') for r in (pumps or [])}
    def object_id(raw, prefix, known):
        candidates = [v for v in (raw, prefix + raw) if v in known]
        return candidates[0] if len(candidates) == 1 else (None if len(candidates) > 1 else (raw if raw.startswith(prefix) else prefix + raw))
    if geometry_path:
        raw = _Path(geometry_path).read_bytes()
        for encoding in ('utf-8-sig','cp1251'):
            try: text=raw.decode(encoding); break
            except UnicodeDecodeError: pass
        else: raise ValueError('Не удалось определить кодировку INP')
        section=''; verts={}; links=[]; duplicate_nodes=set()
        for line in text.splitlines():
            line=line.split(';',1)[0].strip()
            if line.startswith('[') and ']' in line: section=line[1:line.index(']')].upper(); continue
            a=line.split()
            if len(a)<3: continue
            try:
                if section=='COORDINATES':
                    key=object_id(a[0], 'c', node_ids)
                    if key is None: continue
                    if key in geo['nodes']: duplicate_nodes.add(key)
                    else: geo['nodes'][key]=[float(a[1]),float(a[2])]
                elif section=='VERTICES': verts.setdefault(a[0],[]).append([float(a[1]),float(a[2])])
                elif section in ('PIPES','PUMPS','VALVES'): links.append((section,a[0],a[1],a[2]))
            except ValueError: continue
        for key in duplicate_nodes: geo['nodes'].pop(key, None)
        if duplicate_nodes: print(f'⚠️ Неоднозначные координаты для {len(duplicate_nodes)} ID исключены со схемы.')
        for typ,ident,a,b in links:
            p1=geo['nodes'].get(object_id(a, 'c', node_ids)); p2=geo['nodes'].get(object_id(b, 'c', node_ids))
            if p1 and p2: geo['links'].append([object_id(ident, {'PIPES':'e','PUMPS':'p','VALVES':'v'}[typ], pipe_ids if typ=='PIPES' else pump_ids),[p1]+verts.get(ident,[])+[p2]])
        geo['file']=_Path(geometry_path).name
        matched=sum(r['node_id'] in geo['nodes'] for r in (nodes or []))
        print(f'Геометрия: совпало узлов {matched} из {len(nodes or [])}; проверьте соответствие моделей.')
    if facts_for_warning is not None:
        _pmin=number((facts_for_warning.get('nodes_summary') or {}).get('free_pressure_min_m'))
        if _pmin is not None and abs(_pmin) > 1000:
            warning=('<div role="alert" style="margin:20px 0;padding:16px;background:#fff7ed;border:1px solid #f97316;border-radius:10px;color:#7c2d12">'
                '<strong>Проверка качества расчёта:</strong> обнаружен экстремальный свободный напор '
                + _html.escape(str(_pmin)) + ' м. Причина не установлена; агрегаты дефицита и вакуума могут включать недостоверные значения. '
                'До выбора мероприятий проверьте расчёт и исходные значения. Порог 1000 м — служебный индикатор, не норматив.</div>')
            html_doc=html_doc.replace('<section class="ai-report-card">',warning+'<section class="ai-report-card">',1)
    payload=_json.dumps({'tables':tables,'geo':geo,'charts':chart_payload or {}},ensure_ascii=False,allow_nan=False).replace('<','\\u003c')
    insert='''<style>.extra {margin:24px 0;padding:22px;background:white;border:1px solid #d9e4ef;border-radius:14px}.extra button {padding:9px;margin:4px;border:1px solid #94a3b8;border-radius:8px;background:#eff6ff;cursor:pointer}.extra input {padding:9px}.extra-scroll {overflow:auto;max-height:420px}.extra table {border-collapse:collapse;font-size:12px}.extra td,.extra th {border:1px solid #e2e8f0;padding:6px;white-space:nowrap}.extra canvas {width:100%;height:480px;border:1px solid #cbd5e1;background:#f8fafc}.extra small {color:#475569}.object-link {background:transparent;border:0;border-bottom:1px dashed #0284c7;color:#0369a1;font:inherit;cursor:pointer;padding:0 2px}.object-link:hover,.object-link:focus {color:#7c3aed;text-decoration:underline}.extra .object-link {background:transparent;border:0;border-bottom:1px dashed #0284c7;padding:0 2px;margin:0}.map-controls {display:flex;gap:4px;align-items:center;flex-wrap:wrap;margin:10px 0}#networkMap {touch-action:none;cursor:grab}#networkMap:active {cursor:grabbing}</style>
<section class="extra"><h2>Полные расчётные таблицы</h2><p>Состав: дефицит и вакуум; трубы с баллом &lt;80 или Vmax &lt;0,3 / &gt;1,5 м/с; все работающие насосы. Малый Vmax — индикатор, а не доказательство застоя. Отсутствующие данные не считаются нулём.</p><div id="extraButtons"></div><input id="extraSearch" placeholder="Поиск по строке"><div class="extra-scroll"><table id="extraTable"></table></div><button id="extraPrev">Назад</button><span id="extraPage"></span><button id="extraNext">Вперёд</button></section>
<section class="extra"><h2>Дополнительные графики расчётных данных</h2><div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(280px,1fr));gap:20px"><div><h3>Распределение труб по скорости</h3><canvas id="velocityExtra" style="height:250px"></canvas></div><div><h3>Длина труб по диапазонам диаметра</h3><canvas id="diameterExtra" style="height:250px"></canvas></div></div><small>Скоростные категории рассчитаны по максимальной скорости за период; при отсутствии исходных рядов выводы ограничены. Диаметр — протяжённость, км.</small></section>
<section class="extra" id="mapSection"><h2>Схема сети</h2><p id="mapInfo" role="status" aria-live="polite"></p><small>Схема в исходных координатах модели, без географической подложки. Красным показаны объекты с расчётными отклонениями. Колесо мыши — масштаб, перетаскивание — перемещение.</small><div class="map-controls"><button type="button" id="mapPlus" title="Увеличить">＋</button><button type="button" id="mapMinus" title="Уменьшить">−</button><button type="button" id="mapReset">Вся сеть</button><input id="mapSearch" placeholder="ID объекта, например c123" aria-label="ID объекта"><button type="button" id="mapFind">Найти на схеме</button></div><canvas id="networkMap" aria-label="Интерактивная схема сети"></canvas></section>
<script>const EXTRA=__PAYLOAD__;let selected='nodes',page=0;const labels={nodes:'Проблемные узлы',pipes:'Проблемные трубы',pumps:'Работающие насосы'};const buttons=document.getElementById('extraButtons');
for(const k of Object.keys(labels)){let b=document.createElement('button');b.textContent='Посмотреть все: '+labels[k];b.onclick=()=>{selected=k;page=0;drawTable()};buttons.appendChild(b);let d=document.createElement('button');d.textContent='Скачать CSV: '+labels[k];d.onclick=()=>download(k);buttons.appendChild(d)}
function download(k){let rows=EXTRA.tables[k];if(rows===null){alert('Полная сводка отсутствует: повторите обработку CSV.');return}let keys=rows.length?Object.keys(rows[0]):[];let q=x=>'"'+String(x??'').replace(/"/g,'""')+'"';let txt='\\ufeff'+[keys,...rows.map(r=>keys.map(k=>r[k]))].map(r=>r.map(q).join(';')).join('\\r\\n');let url=URL.createObjectURL(new Blob([txt],{type:'text/csv;charset=utf-8'}));let a=document.createElement('a');a.href=url;a.download=k+'.csv';a.click();setTimeout(()=>URL.revokeObjectURL(url),1000)}
function drawTable(){let rows=EXTRA.tables[selected], table=document.getElementById('extraTable');table.replaceChildren();if(rows===null){document.getElementById('extraPage').textContent='Сводка отсутствует — запустите обработку заново';return}let term=document.getElementById('extraSearch').value.toLowerCase();let matches=rows.filter(r=>Object.values(r).some(x=>String(x??'').toLowerCase().includes(term)));let pages=Math.max(1,Math.ceil(matches.length/100));page=Math.min(page,pages-1);document.getElementById('extraPage').textContent=` ${page+1}/${pages} · найдено ${matches.length} `;if(!matches.length)return;let keys=Object.keys(matches[0]);let head=table.createTHead().insertRow();keys.forEach(k=>{let t=document.createElement('th');t.textContent=k;head.appendChild(t)});let body=table.createTBody();matches.slice(page*100,page*100+100).forEach(r=>{let tr=body.insertRow();keys.forEach(k=>{const td=tr.insertCell(),v=r[k]??'';if(['node_id','pipe_id','pump_id'].includes(k)&&v){const button=document.createElement('button');button.type='button';button.className='object-link';button.dataset.mapId=String(v);button.title='Показать на схеме';button.textContent=String(v);td.appendChild(button)}else td.textContent=v})})}
document.getElementById('extraSearch').oninput=()=>{page=0;drawTable()};document.getElementById('extraPrev').onclick=()=>{page=Math.max(0,page-1);drawTable()};document.getElementById('extraNext').onclick=()=>{page++;drawTable()};drawTable();
const networkCanvas = document.getElementById('networkMap');
const networkInfo = document.getElementById('mapInfo');
const networkSection = document.getElementById('mapSection');
const networkGeo = EXTRA.geo || {nodes:{},links:[]};
const linkLookup = new Map(), duplicateLinks = new Set();
for (const [id, points] of (networkGeo.links || [])) {
  if (!id) continue;
  if (linkLookup.has(id)) duplicateLinks.add(id);
  else linkLookup.set(id, points);
}
const networkView = {scale:1, tx:0, ty:0, baseScale:1, selected:null, selectedPoints:null, activeUntil:0, dragging:false};
const badPipes = new Set((EXTRA.tables.pipes || []).map(r=>r.pipe_id));
const mapCache = document.createElement('canvas');
function mapSize(){return {w:Math.max(1, networkCanvas.clientWidth),h:Math.max(1, networkCanvas.clientHeight)};}
function mapResize() {
  const {w,h}=mapSize(), dpr=window.devicePixelRatio || 1;
  networkCanvas.width=Math.round(w*dpr); networkCanvas.height=Math.round(h*dpr);
  mapCache.width=networkCanvas.width; mapCache.height=networkCanvas.height;
}
function screenPoint(p){return [p[0]*networkView.scale+networkView.tx, networkView.ty-p[1]*networkView.scale];}
function redrawBackground(){
  const {w,h}=mapSize(), dpr=window.devicePixelRatio||1, ctx=mapCache.getContext('2d');
  ctx.setTransform(dpr,0,0,dpr,0,0);ctx.clearRect(0,0,w,h);ctx.lineWidth=1;
  for (const [id,points] of (networkGeo.links||[])) {
    if(!points || points.length<2)continue;ctx.beginPath();
    points.forEach((p,i)=>{let [x,y]=screenPoint(p);if(i)ctx.lineTo(x,y);else ctx.moveTo(x,y)});
    ctx.strokeStyle=badPipes.has(id)?'#dc2626':'#aab9c7';ctx.lineWidth=badPipes.has(id)?1.6:0.8;ctx.stroke();
  }
  ctx.fillStyle='#e11d48';
  for(const r of (EXTRA.tables.nodes||[])) {const p=(networkGeo.nodes||{})[r.node_id];if(!p)continue;const [x,y]=screenPoint(p);if(x>=0&&x<=w&&y>=0&&y<=h){ctx.beginPath();ctx.arc(x,y,2.3,0,Math.PI*2);ctx.fill();}}
  redrawOverlay();
}
function redrawOverlay(){
  const dpr=window.devicePixelRatio||1,ctx=networkCanvas.getContext('2d');ctx.setTransform(1,0,0,1,0,0);ctx.clearRect(0,0,networkCanvas.width,networkCanvas.height);ctx.drawImage(mapCache,0,0);
  if(!networkView.selectedPoints)return;
  ctx.setTransform(dpr,0,0,dpr,0,0);const points=networkView.selectedPoints;
  const blink=Date.now()<networkView.activeUntil && Math.floor(Date.now()/400)%2===0;
  ctx.strokeStyle=blink?'#facc15':'#7c3aed';ctx.fillStyle=blink?'#facc15':'#7c3aed';ctx.lineWidth=blink?7:5;
  if(points.length>1){ctx.beginPath();points.forEach((p,i)=>{let [x,y]=screenPoint(p);if(i)ctx.lineTo(x,y);else ctx.moveTo(x,y)});ctx.stroke();}
  else {const [x,y]=screenPoint(points[0]);ctx.beginPath();ctx.arc(x,y,blink?9:6,0,Math.PI*2);ctx.fill();ctx.strokeStyle='#fff';ctx.lineWidth=2;ctx.stroke();}
}
function mapFit(){
  const pts=Object.values(networkGeo.nodes||{}).filter(p=>p&&Number.isFinite(p[0])&&Number.isFinite(p[1]));
  if(!pts.length){networkInfo.textContent='Координаты не найдены; схема недоступна.';return;}
  mapResize();let xmin=Infinity,xmax=-Infinity,ymin=Infinity,ymax=-Infinity;
  pts.forEach(([x,y])=>{xmin=Math.min(xmin,x);xmax=Math.max(xmax,x);ymin=Math.min(ymin,y);ymax=Math.max(ymax,y)});
  const {w,h}=mapSize(),scale=Math.min((w-40)/Math.max(xmax-xmin,1),(h-40)/Math.max(ymax-ymin,1));
  networkView.baseScale=scale;networkView.scale=scale;networkView.tx=w/2-(xmin+xmax)*scale/2;networkView.ty=h/2+(ymin+ymax)*scale/2;
  networkInfo.textContent=`Узлов с координатами: ${pts.length}; связей: ${(networkGeo.links||[]).length}. Выберите ID или воспользуйтесь поиском.`;redrawBackground();
}
function focusMapObject(id){
  id=String(id||'').trim();const nodes=networkGeo.nodes||{};
  networkSection.scrollIntoView({behavior:'smooth',block:'start'});
  if(duplicateLinks.has(id)){networkInfo.textContent=`${id}: неоднозначный ID в геометрии; объект не выделен.`;return false;}
  const node=Object.prototype.hasOwnProperty.call(nodes,id)?nodes[id]:null;
  const link=linkLookup.get(id);
  if(node&&link){networkInfo.textContent=`${id}: ID неоднозначен; объект не выделен.`;return false;}
  const pts=node?[node]:link;
  if(!pts||!pts.length){networkInfo.textContent=`${id}: нет однозначного совпадения или координат в INP.`;return false;}
  let xmin=Infinity,xmax=-Infinity,ymin=Infinity,ymax=-Infinity;
  pts.forEach(([x,y])=>{xmin=Math.min(xmin,x);xmax=Math.max(xmax,x);ymin=Math.min(ymin,y);ymax=Math.max(ymax,y)});
  const {w,h}=mapSize(),spanX=xmax-xmin,spanY=ymax-ymin;
  let scale=networkView.baseScale*12;
  if(pts.length>1 && (spanX>0||spanY>0)) scale=Math.min((w*.55)/Math.max(spanX,1),(h*.55)/Math.max(spanY,1),networkView.baseScale*40);
  scale=Math.max(networkView.baseScale,scale);
  networkView.scale=scale;networkView.tx=w/2-(xmin+xmax)*scale/2;networkView.ty=h/2+(ymin+ymax)*scale/2;
  networkView.selected=id;networkView.selectedPoints=pts;networkView.activeUntil=Date.now()+6000;
  networkInfo.textContent=`Выбран объект ${id}. Подсветка мигает 6 секунд; точные координаты остаются в исходной системе модели.`;
  redrawBackground();return true;
}
function zoomMap(factor, x, y){
  const old=networkView.scale, next=Math.max(networkView.baseScale*.25,Math.min(networkView.baseScale*100,old*factor));
  const wx=(x-networkView.tx)/old, wy=(networkView.ty-y)/old;networkView.scale=next;networkView.tx=x-wx*next;networkView.ty=y+wy*next;redrawBackground();
}
networkCanvas.addEventListener('wheel',ev=>{ev.preventDefault();const rect=networkCanvas.getBoundingClientRect();zoomMap(ev.deltaY<0?1.25:.8,ev.clientX-rect.left,ev.clientY-rect.top)},{passive:false});
networkCanvas.addEventListener('pointerdown',ev=>{networkView.dragging=true;networkView.x=ev.clientX;networkView.y=ev.clientY;networkCanvas.setPointerCapture(ev.pointerId);});
networkCanvas.addEventListener('pointermove',ev=>{if(!networkView.dragging)return;networkView.tx+=ev.clientX-networkView.x;networkView.ty+=ev.clientY-networkView.y;networkView.x=ev.clientX;networkView.y=ev.clientY;redrawBackground();});
networkCanvas.addEventListener('pointerup',()=>{networkView.dragging=false});
networkCanvas.addEventListener('pointercancel',()=>{networkView.dragging=false});
document.getElementById('mapPlus').onclick=()=>{const {w,h}=mapSize();zoomMap(1.4,w/2,h/2)};
document.getElementById('mapMinus').onclick=()=>{const {w,h}=mapSize();zoomMap(1/1.4,w/2,h/2)};
document.getElementById('mapReset').onclick=()=>{networkView.selectedPoints=null;mapFit()};
document.getElementById('mapFind').onclick=()=>focusMapObject(document.getElementById('mapSearch').value);
document.getElementById('mapSearch').addEventListener('keydown',ev=>{if(ev.key==='Enter')focusMapObject(ev.target.value)});
document.addEventListener('click',ev=>{const b=ev.target.closest('[data-map-id]');if(b){ev.preventDefault();focusMapObject(b.dataset.mapId);}});
function makeIdButton(id){let b=document.createElement('button');b.type='button';b.className='object-link';b.dataset.mapId=id;b.title='Показать на схеме';b.textContent=id;return b;}
// Существующие top-таблицы и фрагменты отчёта с ID в <code>.
for(const code of document.querySelectorAll('.tables-section code, .ai-content code')){const id=code.textContent.trim();if(/^[cepv]\\d+$/i.test(id))code.replaceWith(makeIdButton(id));}
// Некоторые модели пишут ID обычным текстом — добавляем переход, не переписывая остальную разметку.
for(const container of document.querySelectorAll('.ai-content')){
  const walker=document.createTreeWalker(container,NodeFilter.SHOW_TEXT);const nodes=[];while(walker.nextNode())nodes.push(walker.currentNode);
  for(const node of nodes){if(node.parentElement && node.parentElement.closest('button,a,code,script,style'))continue;const text=node.nodeValue, re=/\\b[cepv]\\d+\\b/gi;if(!re.test(text))continue;re.lastIndex=0;let frag=document.createDocumentFragment(),last=0,m;while((m=re.exec(text))){frag.appendChild(document.createTextNode(text.slice(last,m.index)));frag.appendChild(makeIdButton(m[0]));last=m.index+m[0].length}frag.appendChild(document.createTextNode(text.slice(last)));node.replaceWith(frag);}
}
window.addEventListener('resize',mapFit);
requestAnimationFrame(mapFit);
setInterval(()=>{if(networkView.selectedPoints&&Date.now()<networkView.activeUntil+500)redrawOverlay()},400);

function extraBars(id, vals, units){let c=document.getElementById(id), ctx=c.getContext('2d');c.width=c.clientWidth*devicePixelRatio;c.height=c.clientHeight*devicePixelRatio;ctx.scale(devicePixelRatio,devicePixelRatio);let w=c.clientWidth,h=c.clientHeight,entries=Object.entries(vals||{}).filter(x=>typeof x[1]==='number'&&Number.isFinite(x[1]));ctx.font='12px sans-serif';if(!entries.length){ctx.fillStyle='#64748b';ctx.fillText('Нет данных',12,30);return}let max=Math.max(...entries.map(x=>x[1]),1),row=(h-20)/entries.length;entries.forEach(([k,v],i)=>{let y=i*row+10;ctx.fillStyle='#334155';ctx.fillText(k,8,y+14);ctx.fillStyle='#0284c7';ctx.fillRect(8,y+20,(w-100)*v/max,Math.max(3,Math.min(14,row-26)));ctx.fillStyle='#0f172a';ctx.fillText(String(v)+' '+units,Math.min(w-80,15+(w-100)*v/max),y+31)})}
requestAnimationFrame(()=>{extraBars('velocityExtra',EXTRA.charts.velocity_bins,'шт.');extraBars('diameterExtra',EXTRA.charts.diameter_km_bins,'км')});
</script>'''.replace('__PAYLOAD__',payload)
    return html_doc.replace('</body>',insert+'</body>')


import gc
import json
import time
import unicodedata
from pathlib import Path
from IPython.display import HTML, display

# ==============================================================================
# НАСТРОЙКИ ЗАПУСКА КНОПКИ «АНАЛИЗ ИИ»
# ==============================================================================
# Укажите имя города из data/ (например: "куляб", "тенино", "североморск", "лесное", "модель 10", "модель 11")
# либо полный путь к любой папке с необработанными CSV ПК «Аквабит»:
TARGET_CITY_OR_PATH = "модель семей"
# Необязательно: полный путь к текстовой INP-модели; TXT также допустим.
GEOMETRY_INP_PATH = None  # None: найти единственный *.inp в папке города
# GGUF: загрузка из Hugging Face только при первом запуске.
GGUF_REPO = "unsloth/Qwen3.5-4B-GGUF"
GGUF_FILENAME = "Qwen3.5-4B-Q4_K_M.gguf"
LLAMA_N_GPU_LAYERS = 32  # 0 = CPU; увеличивать только после проверки свободной VRAM
LLAMA_CONTEXT = 65536  # общий бюджет: промпт + ответ; требует дополнительной RAM/VRAM
LLAMA_BACKEND = "cuda"  # надёжный старт; "cuda" для экспериментальной загрузки на NVIDIA
LLAMA_SERVER_PATH = None  # иначе бинарник Windows скачивается с официального GitHub llama.cpp
INFERENCE_MAX_NEW_TOKENS = 24576  # максимум, не целевой размер отчёта

CUSTOM_ENGINEER_QUESTION = (
    "Сформируй полный инженерный отчёт ИИ-диагностики по расчётной модели ПК «Аквабит» по всем 5 разделам."
)

FAST_SYSTEM_PROMPT_V2 = (
    "Ты инженер-аналитик водопроводных сетей ПК «Аквабит». Пиши на русском в Markdown, без приветствий.\n"
    "Не описывай файлы, CSV, JSON, названия колонок, структуру выгрузки, очистку или ход обработки; "
    "допустимо кратко указать недостаточность или недостоверность расчётных данных, когда это влияет на вывод.\n"
    "Требуемое давление указано в метрах водяного столба. Не утверждай ошибку единиц или высот без доказательств. "
    "Высотная отметка узла — отметка местности, а не высота здания. Нулевые строки и транзитные узлы "
    "отфильтрованы только для аналитики, не удалены из гидравлического расчёта.\n"
    "Приводи только данные и ID из входа; не выдумывай значения, источник причины, норматив, КПД, расчёт экономии или аварии. "
    "Крайне неправдоподобное значение напора — предупреждение о качестве расчёта, а не доказанный вакуум, "
    "ошибка единиц или ошибка конкретного узла. Дефицит и вакуум, рассчитанные с такими значениями, "
    "считай предварительными до проверки; не обещай устранение всех таких узлов.\n"
    "Не называй объём воды в трубах пропускной способностью. Сумма подачи насосов не равна автоматически "
    "подаче в сеть и не позволяет определить потери: насосы могут стоять последовательно. "
    "Не считай отсутствие измерений нулём. Нулевая подача клапана сама по себе не доказывает его закрытие; "
    "реверс сам по себе не авария. Максимальная скорость менее 0.3 м/с лишь индикатор малой скорости. "
    "Одна скорость не доказывает гидроудар; QH без КПД не доказывает BEP, перегрузку или энергоэффективность. "
    "Относительный балл шероховатости: меньше 80 хуже, 80 нормальный, 100 хороший; балл не подтверждает физический износ.\n"
    "Структура строго: ## 1. Водный баланс; ## 2. Напоры; ## 3. Трубопроводы; "
    "## 4. Насосы и арматура; ## 5. Приоритетные инженерные мероприятия. "
    "Разделы 1–4 кратко: факт, значение, достоверность и при необходимости гипотеза. "
    "Раздел 5 — наиболее подробный и главный: отдельные приоритетные мероприятия по подтверждённым проблемам "
    "и условные варианты для неподтверждённых. Для каждого: цель, конкретные объекты только если ID имеются, "
    "обоснование фактами, что и как проверить, последовательность действий в модели, критерий проверки результата, "
    "ограничения и что нельзя утверждать без измерений. Укажи очередность, зависимости, сравнение сценариев "
    "и риски вмешательства. Не назначай замену труб, установку клапанов или реконструкцию как доказанное решение "
    "до обследования и проверочного расчёта. Не придумывай пороги, срок, стоимость, экономию или масштаб. "
    "Излагай предметно и понятно, без повторов и описания входных данных."
)


def _format_digest_for_infer(facts: dict, charts: dict) -> str:
    """Весь подготовленный паспорт и выбранные наборы данных графиков; не весь CSV R0..R1439."""
    chart_keys = (
        'hours_labels', 'diurnal_demand_m3_h', 'diurnal_pump_flow_m3_h',
        'diurnal_valve_flow_m3_h', 'diurnal_free_pressure_percentiles_m',
        'pressure_bins', 'velocity_bins', 'roughness_km_bins', 'diameter_km_bins',
        'demand_schedules_24h', 'qh_curves')
    prepared_charts = {key: charts.get(key) for key in chart_keys}
    _minimum = (facts.get('nodes_summary') or {}).get('free_pressure_min_m')
    try: _extreme_pressure = _minimum is not None and abs(float(_minimum)) > 1000
    except (TypeError, ValueError): _extreme_pressure = False
    _data_caution = ("ПРЕДУПРЕЖДЕНИЕ: экстремальный свободный напор; причина неизвестна. "
        "Агрегаты дефицита и вакуума предварительны и могут включать недостоверные точки. "
        "Порог 1000 м — служебная проверка, не норматив.\n" if _extreme_pressure else "")
    return (
        _data_caution + "РАСЧЁТНЫЕ ФАКТЫ JSON:\n" + json.dumps(facts, ensure_ascii=False, default=str, separators=(',', ':')) +
        "\nДАННЫЕ ГРАФИКОВ JSON:\n" + json.dumps(prepared_charts, ensure_ascii=False, default=str, separators=(',', ':')) +
        "\nДанные графиков — расчётные агрегаты, а не изображения или измерения. "
        "Это не полные исходные CSV: таблицы объектов в паспорте могут содержать только топ выборку, "
        "а payload QH и графиков ограничен при подготовке данных. Не считай их полным перечнем. "
        "В некоторых QH curve_points — интерполяция или предположенная парабола, а не паспортные измерения. "
        "Пустое или null означает нет данных. Не определяй BEP и электрическую энергию без данных о КПД. "
        "Сумма подачи насосов не обязательно равна отпуску из-за возможной повторной перекачки. "
        "Реверс не обязательно авария, низкая максимальная скорость не доказывает застой. "
        "Причины, кавитация и гидроудар — только гипотезы до проверки."
    )


# GGUF и llama-server изолированы от обучения/старых адаптеров.
def _download_llama_server(backend):
    import os, sys, zipfile, requests
    if LLAMA_SERVER_PATH:
        exe = Path(LLAMA_SERVER_PATH)
        if not exe.is_file(): raise FileNotFoundError(f"llama-server не найден: {exe}")
        return exe
    if sys.platform != 'win32':
        raise RuntimeError("Укажите LLAMA_SERVER_PATH для установленного llama-server этой ОС")
    target = MODELS_DIR / 'llama_cpp' / backend
    exe = target / 'llama-server.exe'
    if exe.exists(): return exe
    target.mkdir(parents=True, exist_ok=True)
    api = 'https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=10'
    response = requests.get(api, timeout=45); response.raise_for_status()
    releases = response.json()
    match = None
    for release in releases:
        assets=release.get('assets', [])
        suffix = '-bin-win-cuda-12.4-x64.zip' if backend == 'cuda' else '-bin-win-cpu-x64.zip'
        # cudart-llama-bin-win-cuda-12.4-x64.zip тоже имеет такой суффикс,
        # но содержит только библиотеки CUDA, а не llama-server.exe.
        # Поэтому выбираем именно архив llama-bNNNN-bin-..., а не cudart.
        binary = next((a for a in assets if a['name'].startswith('llama-') and a['name'].endswith(suffix)), None)
        runtime = next((a for a in assets if a['name'] == 'cudart-llama-bin-win-cuda-12.4-x64.zip'), None) if backend == 'cuda' else None
        if binary and (backend != 'cuda' or runtime):
            match = (binary, runtime)
            print(f"llama.cpp release {release.get('tag_name', '?')}: {binary['name']}")
            break
    if match is None: raise RuntimeError('Не найден совместимый архив llama.cpp на GitHub; задайте LLAMA_SERVER_PATH')
    for asset in [a for a in match if a is not None]:
        url=asset['browser_download_url']
        if not url.startswith('https://github.com/ggml-org/llama.cpp/releases/download/'):
            raise ValueError('Неожиданный адрес архива llama.cpp')
        archive=target / asset['name']
        print(f"Скачивание {asset['name']} ({asset['size']/1048576:.0f} МБ)...")
        with requests.get(url, stream=True, timeout=(30,300)) as r:
            r.raise_for_status()
            with archive.open('wb') as out:
                for part in r.iter_content(1024*1024):
                    if part: out.write(part)
        with zipfile.ZipFile(archive) as z:
            # Проверяем ПЕРЕД извлечением: если серверного exe нет, сообщаем имя
            # реального архива и не подменяем его библиотеками CUDA.
            if asset is match[0] and not any(
                member.filename.replace('\\', '/').split('/')[-1].lower() == 'llama-server.exe'
                for member in z.infolist() if not member.is_dir()
            ):
                raise RuntimeError(f"Архив {asset['name']} не содержит llama-server.exe; выберите другую сборку через LLAMA_SERVER_PATH")
            # Не доверяем путям из архива, не извлекаем во вложенные каталоги.
            for member in z.infolist():
                name = Path(member.filename.replace('\\','/')).name
                if not member.is_dir() and name and name.lower().endswith(('.exe','.dll')):
                    with z.open(member) as src, (target/name).open('wb') as out:
                        import shutil; shutil.copyfileobj(src,out)
        archive.unlink()
    if not exe.exists(): raise FileNotFoundError(f"В архиве отсутствует {exe.name}")
    return exe


def _stream_local_gguf(messages):
    import os, socket, subprocess, tempfile, time, requests
    from huggingface_hub import hf_hub_download
    backend = LLAMA_BACKEND.lower()
    if backend not in ('cpu','cuda'): raise ValueError("LLAMA_BACKEND: 'cpu' или 'cuda'")
    exe = _download_llama_server(backend)
    print(f"Модель {GGUF_REPO}/{GGUF_FILENAME}: при первом запуске будет скачано несколько ГБ.")
    model_file=hf_hub_download(repo_id=GGUF_REPO, filename=GGUF_FILENAME,
        local_dir=str(MODELS_DIR / 'gguf' / GGUF_REPO.replace('/','--')))
    with socket.socket() as sock:
        sock.bind(('127.0.0.1',0)); port=sock.getsockname()[1]
    base=f'http://127.0.0.1:{port}'
    args=[str(exe), '-m', str(model_file), '--host','127.0.0.1','--port',str(port),
          '-c',str(LLAMA_CONTEXT), '-np','1', '-ngl',str(LLAMA_N_GPU_LAYERS if backend=='cuda' else 0), '--no-kv-offload', '--no-context-shift', '--jinja']
    chunks=[]; finish=None; count=None; start=time.perf_counter()
    # Сервер завершается даже при ошибке, чтобы не оставлять модель в памяти.
    with tempfile.TemporaryFile(mode='w+t',encoding='utf-8') as log:
        proc=subprocess.Popen(args,stdout=log,stderr=subprocess.STDOUT,
                              creationflags=getattr(subprocess,'CREATE_NO_WINDOW',0))
        try:
            deadline=time.monotonic()+300
            while time.monotonic()<deadline:
                if proc.poll() is not None:
                    log.seek(0); raise RuntimeError('llama-server завершился:\n'+log.read()[-6000:])
                try:
                    r=requests.get(base+'/health',timeout=2)
                    if r.status_code==200: break
                except requests.RequestException: pass
                time.sleep(1)
            else: raise TimeoutError('llama-server не запустился за 300 сек')
            # Токенизируем полный чат с шаблоном модели, не обрезая промпт.
            available = LLAMA_CONTEXT - INFERENCE_MAX_NEW_TOKENS - 256
            if available <= 0:
                raise ValueError('LLAMA_CONTEXT слишком мал для выбранного лимита ответа и запаса.')
            req = requests.post(base+'/apply-template', json={'messages': messages,
                'chat_template_kwargs': {'enable_thinking': False}}, timeout=60)
            if not req.ok:
                raise RuntimeError(f'Ошибка /apply-template HTTP {req.status_code}: {req.text[:1200]}')
            rendered = req.json().get('prompt')
            if not isinstance(rendered, str): raise RuntimeError('llama-server не вернул шаблон промпта')
            tok = requests.post(base+'/tokenize', json={'content':rendered, 'add_special':True}, timeout=60)
            if not tok.ok: raise RuntimeError(f'Ошибка /tokenize HTTP {tok.status_code}: {tok.text[:1200]}')
            token_ids = tok.json().get('tokens')
            if not isinstance(token_ids,list): raise RuntimeError('llama-server не вернул список токенов')
            n_prompt=len(token_ids)
            print(f'Контекст: полный подготовленный промпт {n_prompt} токенов; максимум входа {available}, резерв ответа {INFERENCE_MAX_NEW_TOKENS}.')
            if n_prompt > available:
                raise ValueError(f'Промпт не помещается без сокращения: {n_prompt} > {available} токенов. Увеличьте LLAMA_CONTEXT или уменьшите INFERENCE_MAX_NEW_TOKENS; потребуется больше памяти.')
            payload={'model':'local','messages':messages,'stream':True,'max_tokens':INFERENCE_MAX_NEW_TOKENS,
                'temperature':0.7,'top_p':0.8,'chat_template_kwargs':{'enable_thinking':False},
                'stream_options':{'include_usage':True}}
            print('\n🤖 Генерация отчёта Qwen3.5-4B:\n'+'='*70)
            with requests.post(base+'/v1/chat/completions',json=payload,stream=True,timeout=(15,900)) as r:
                if not r.ok: raise RuntimeError(f"llama-server HTTP {r.status_code}: {r.text[:3000]}")
                r.encoding = 'utf-8'  # SSE всегда UTF-8, даже без charset в заголовках
                for line in r.iter_lines(decode_unicode=True):
                    if not line or not line.startswith('data:'): continue
                    data=line[5:].strip()
                    if data=='[DONE]': break
                    event=json.loads(data)
                    if event.get('error'): raise RuntimeError(str(event['error']))
                    usage=event.get('usage') or {}
                    if usage.get('completion_tokens') is not None: count=usage['completion_tokens']
                    for choice in event.get('choices',[]):
                        piece=(choice.get('delta') or {}).get('content') or ''
                        if piece: chunks.append(piece);print(piece,end='',flush=True)
                        if choice.get('finish_reason'): finish=choice['finish_reason']
            if finish is None:
                raise RuntimeError('Поток завершился без finish_reason: отчёт может быть неполным')
            return ''.join(chunks), finish, count, round(time.perf_counter()-start,2)
        finally:
            proc.terminate()
            try: proc.wait(timeout=10)
            except subprocess.TimeoutExpired: proc.kill();proc.wait()

def run_aquabit_ai_analysis_live(target_city_or_path: str | Path, question: str) -> Path:
    t_total_start = time.perf_counter()
    cand_path = Path(target_city_or_path)
    cname_norm = unicodedata.normalize("NFC", str(target_city_or_path)).strip()

    if cand_path.exists() and cand_path.is_dir():
        print(f"⚙️ [1/3] Сверхбыстрая обработка папки CSV «{cand_path.name}»...")
        facts, chart_payload = process_aquabit_city_fast(cand_path, cleaned_root_dir=DATA_CLEANED_DIR, top_k=10)
    elif cname_norm in ALL_CITY_FACTS and cname_norm in ALL_CITY_CHARTS:
        facts, chart_payload = ALL_CITY_FACTS[cname_norm], ALL_CITY_CHARTS[cname_norm]
        print(f"⚡ [1/3] Подгружен паспорт и графики города «{cname_norm}» (обработка CSV заняла {facts.get('processing_time_sec', 0)} сек)")
    elif (RAW_DATA_DIR / cname_norm).exists():
        print(f"⚙️ [1/3] Сверхбыстрая обработка сырой папки «{RAW_DATA_DIR / cname_norm}»...")
        facts, chart_payload = process_aquabit_city_fast(RAW_DATA_DIR / cname_norm, cleaned_root_dir=DATA_CLEANED_DIR, top_k=10)
    elif ALL_CITY_FACTS:
        raise FileNotFoundError(f"Модель {target_city_or_path!r} не найдена; доступно: {list(ALL_CITY_FACTS)}")
    else:
        raise FileNotFoundError(f"Не найдены CSV-данные ни в {RAW_DATA_DIR.resolve()}, ни по пути {target_city_or_path}")

    cname = facts["city_name"]
    digest_str = _format_digest_for_infer(facts, chart_payload)
    user_prompt = f"Запрос анализа ПК «Аквабит»: {question}\n\nСводка расчёта Аквабит:\n{digest_str}"
    messages = [
        {"role": "system", "content": build_ontology_ai_system_prompt(load_hydraulic_ontology(RAW_DATA_DIR / "aquabit_hydraulic_ontology_v3.json"))},
        {"role": "user", "content": user_prompt},
    ]
    raw_full_text, _finish_reason, _out_count, gen_sec = _stream_local_gguf(messages)
    _token_limit_hit = _finish_reason != "stop"
    if _token_limit_hit:
        print("\n⚠️ Достигнут лимит вывода или контекста; отчёт может быть неполным.")
    model_label = GGUF_REPO + " / " + GGUF_FILENAME

    clean_ai_markdown = sanitize_pure_facts_response(raw_full_text)
    # Проверка структуры и повторов: при сбое не выдаём машинный текст за ИИ-вывод.
    import re
    _sections = set(re.findall(r"(?m)^\s*#{1,3}\s*([1-5])\.", clean_ai_markdown))
    _words = re.findall(r"[\wА-Яа-яЁё]+", clean_ai_markdown.lower())
    _grams = [tuple(_words[i:i+12]) for i in range(max(0, len(_words)-11))]
    _loop = len(_grams) > 20 and len(set(_grams)) < len(_grams)*0.8
    if len(_sections) < 2 or _loop or _token_limit_hit or raw_full_text.lstrip().startswith("{"):
        print("\n⚠️ Ответ может быть неполным, повторяющимся или неструктурированным. Он сохранён как черновик; требуется проверка.")
        clean_ai_markdown = ("**Внимание: черновик не прошёл автоматическую проверку структуры или завершения.**\n\n" + clean_ai_markdown)
    from IPython.display import Markdown
    display(Markdown(clean_ai_markdown))

    out_tokens_cnt = _out_count
    tok_per_sec = round(out_tokens_cnt / max(gen_sec, 0.01), 1) if out_tokens_cnt is not None else None
    print("\n" + "=" * 88)
    print(f"⚡ Генерация завершена: {out_tokens_cnt} токенов за {gen_sec} сек ({tok_per_sec} ток/сек)")

    out_html_path = REPORTS_DIR / f"EPANET_AI_Report_{cname}.html"
    _md_path = REPORTS_DIR / f"EPANET_AI_Report_{cname}.md"
    _md_path.write_text(clean_ai_markdown, encoding="utf-8")
    (REPORTS_DIR / f"EPANET_AI_Raw_{cname}.md").write_text(raw_full_text, encoding="utf-8")
    print(f"Markdown и исходный ответ сохранены: {_md_path}")
    
    # Пересчет графиков через Matplotlib Engine и извлечение выбранных ИИ дашбордов
    from aquabit_ai.hydraulic_matplotlib import render_all_recalculated_charts, assemble_adaptive_dashboard_html
    from aquabit_ai.ontology_engine import extract_ai_dashboards_and_content, load_hydraulic_ontology

    facts["chart_payload"] = chart_payload
    all_mpl_plots = render_all_recalculated_charts(facts)
    clean_ai_markdown, parsed_dashboards = extract_ai_dashboards_and_content(clean_ai_markdown)
    chosen_dashboards = [d["id"] for d in parsed_dashboards.get("dashboards", [])]
    print(f"\n💡 ИИ выбрал дашборды для отображения: {chosen_dashboards}")

    # Отображаем приоритетный график прямо в выводе Jupyter Notebook
    top_chart_id = chosen_dashboards[0] if chosen_dashboards else "pump_qh"
    if top_chart_id in all_mpl_plots:
        print(f"📊 Превью приоритетного дашборда ({all_mpl_plots[top_chart_id]['title']}):")
        display(HTML(f'<img src="{all_mpl_plots[top_chart_id]["b64"]}" style="max-width:85%;border-radius:8px;border:1px solid #cbd5e1;box-shadow:0 4px 6px rgba(0,0,0,0.05);"/>'))

    render_standalone_html_report(
        facts=facts,
        chart_payload=chart_payload,
        ai_markdown=clean_ai_markdown,
        output_html_path=out_html_path,
        generation_meta={
            "model_name": f"{model_label} ({tok_per_sec} tok/s)",
            "inference_sec": gen_sec,
        },
    )
    # Дополнение встраивается в HTML без интернета. Старые паспорта требуют повторной обработки.
    _summary_dir = DATA_CLEANED_DIR / cname
    _city_dir = cand_path if cand_path.is_dir() else RAW_DATA_DIR / cname
    _geometry = Path(GEOMETRY_INP_PATH) if GEOMETRY_INP_PATH else None
    if _geometry is None and _city_dir.is_dir():
        _candidates = sorted([p for p in _city_dir.iterdir() if p.is_file() and p.suffix.lower() == '.inp'])
        if len(_candidates) == 1:
            _geometry = _candidates[0]
        elif len(_candidates) > 1:
            raise ValueError(f"Найдено несколько INP для {cname}: {_candidates}; задайте GEOMETRY_INP_PATH явно")
        else:
            print(f"ℹ️ INP не найден в {_city_dir}; схема не построена")
    _charts_for_extra = dict(chart_payload)
    if not facts.get('pipes_summary', {}).get('flow_timeseries_available'):
        _charts_for_extra['velocity_bins'] = None
    _full_html = _report_extension(out_html_path.read_text(encoding="utf-8"), _summary_dir, _geometry, _charts_for_extra, facts)
    out_html_path.write_text(_full_html, encoding="utf-8")
    total_e2e = round(time.perf_counter() - t_total_start, 2)
    print(f"📊 [3/3] Готов интерактивный HTML-отчёт с графиками Кривой QH: {out_html_path.resolve()}")
    print(f"🏁 Полное время цикла «Анализ ИИ»: {total_e2e} сек!")
    display(HTML(f'<div style="padding:12px 16px;background:#ecfdf5;border:1px solid #10b981;border-radius:10px;margin-top:8px">'
                 f'✅ <b>Отчёт сохранён:</b> <code>{out_html_path.resolve()}</code> ({out_html_path.stat().st_size / 1024:.1f} КБ) — '
                 f'откройте его в любом браузере для интерактивного просмотра графиков <b>Кривая QH</b> и печати в PDF!</div>'))
    return out_html_path


REPORT_HTML_FILE = run_aquabit_ai_analysis_live(TARGET_CITY_OR_PATH, CUSTOM_ENGINEER_QUESTION)


⚡ [1/3] Подгружен паспорт и графики города «модель семей» (обработка CSV заняла 16.51 сек)
Модель unsloth/Qwen3.5-4B-GGUF/Qwen3.5-4B-Q4_K_M.gguf: при первом запуске будет скачано несколько ГБ.
Контекст: полный подготовленный промпт 22131 токенов; максимум входа 40704, резерв ответа 24576.

🤖 Генерация отчёта Qwen3.5-4B:
## 1. Водный баланс
*   **Факт:** Суммарная суточная подача насосов составляет 156 372,5 м³/сут, что превышает пиковую часовую нагрузку сети (3607,61 м³/ч ≈ 86 582,64 м³/сут) в 1,8 раза. Пиковая нагрузка приходится на 20:00.
*   **Значение:** Эффективность использования энергии насосов оценивается как низкая (коэффициент полезного действия не учитывается). Существует значительный разрыв между ресурсом насосов и реальным расходом сети.
*   **Достоверность:** Высокая (на основе суммарных данных из JSON).
*   **Гипотеза:** Наличие последовательного подключения насосов (насосы-насосы) или неучтенных потерь в сети. Сумма подачи насосов не является прямой мерой потребления се

## 1. Водный баланс
*   **Факт:** Суммарная суточная подача насосов составляет 156 372,5 м³/сут, что превышает пиковую часовую нагрузку сети (3607,61 м³/ч ≈ 86 582,64 м³/сут) в 1,8 раза. Пиковая нагрузка приходится на 20:00.
*   **Значение:** Эффективность использования энергии насосов оценивается как низкая (коэффициент полезного действия не учитывается). Существует значительный разрыв между ресурсом насосов и реальным расходом сети.
*   **Достоверность:** Высокая (на основе суммарных данных из JSON).
*   **Гипотеза:** Наличие последовательного подключения насосов (насосы-насосы) или неучтенных потерь в сети. Сумма подачи насосов не является прямой мерой потребления сети.

## 2. Напоры
*   **Факт:** В сети зафиксированы 12 узлов с вакуумом (< 0 м) и 194 узла с дефицитом напора (0 м .. < P_треб). Максимальное падение напора до -524,3 м. Максимальный избыток напора (> 60 м) наблюдается на 50 узлах.
*   **Значение:** Критическая ситуация по водоснабжению (вакуум и дефицит) и избыточное давление, требующее балансировки.
*   **Достоверность:** Высокая (на основе данных `free_pressure_min_m` и `deficit_nodes_below_required_count`).
*   **Гипотеза:** Дефицит в верхних этажах (высоты 12-13 эт.) может быть связан с локальными потерями или недостаточной мощностью источников в этой зоне, а не с общей высотой здания.

## 3. Трубопроводы
*   **Факт:** 19 713 труб (92,7%) имеют скорость течения < 0,3 м/с (застой). 8 труб имеют скорость > 3,0 м/с (риск гидроудара). 1111 труб демонстрируют реверс потока. Общая шероховатость сети — 76,03 баллов (ниже нормы), 20 468 труб имеют шероховатость < 78 баллов.
*   **Значение:** Значительная часть сети работает в режиме застоя, что повышает риск коррозии и биологического загрязнения. Наличие реверса указывает на нестабильность режимов работы.
*   **Достоверность:** Высокая (на основе данных `velocity_bins` и `roughness_km_bins`).
*   **Гипотеза:** Застой в крупных трубах (диаметр > 300 мм) может быть следствием их использования для резервирования или неправильного моделирования потребностей в пиковые часы.

## 4. Насосы и арматура
*   **Факт:** Из 127 насосов 35 находятся в резерве/выключенном состоянии. 4 насоса работают в зоне перегрузки (правее номинала, > 1,15 Q_ном), что создает риск кавитации. 5 насосов работают в зоне недогрузки (левее номинала), что указывает на дросселирование.
*   **Значение:** Насосы работают неэффективно. Перегрузка может привести к физическому износу и кавитации, недогрузка — к перерасходу электроэнергии.
*   **Достоверность:** Средняя (на основе `q_to_nominal_ratio` и `operating_zone`, КПД не учтен).
*   **Гипотеза:** Дросселирование насосов может быть связано с наличием задвижек на входе, которые не учитываются в гидравлическом расчёте как активное сопротивление, либо с несоответствием мощности насосов текущей нагрузке сети.

## 5. Приоритетные инженерные мероприятия

### Мероприятие 1: Диагностика и устранение вакуума и дефицита напора
*   **Цель:** Устранить 12 узлов с вакуумом и снизить количество узлов с дефицитом напора до минимума.
*   **Объекты:** Узлы: `c3467991`, `c3468019`, `c3424270`, `c3468009`, `c3468000`, `c3459771`, `c3459773`, `c3439082`, `c3459761` (высокий приоритет по дефициту).
*   **Обоснование:** Расчетные данные показывают критический дефицит напора, особенно на высотных этажах (12-13 эт.).
*   **Что и как проверить:**
    1.  Провести гидравлический анализ сети с учетом всех потерь в трубах и локальных сопротивлений.
    2.  Проверить фактическое положение задвижек на входе в узлы дефицита.
    3.  Измерить реальные напоры в проблемных узлах (особенно `c3467991` с вакуумом -524,3 м).
*   **Последовательность действий:**
    1.  В модели исключить некорректные потери на локальные сопротивления (если они не подтверждены).
    2.  Если дефицит сохраняется, рассмотреть установку дополнительных насосных станций или увеличение давления на входе в проблемную зону.
    3.  Проверить возможность подключения к существующим источникам с более высоким напором.
*   **Критерий проверки:** Отсутствие узлов с вакуумом и дефицитом напора в новом расчете.
*   **Ограничения:** Не назначать замену источников или установку новых насосов без подтверждения необходимости измерением.
*   **Риски:** Внезапное повышение давления может привести к повреждению арматуры в системах с высоким избыточным напором.

### Мероприятие 2: Оптимизация режима работы насосов
*   **Цель:** Свести к минимуму работу насосов в зонах перегрузки и недогрузки, повысить КПД системы.
*   **Объекты:** Насосы: `p3467585`, `p3467665` (перегрузка); `p3381996`, `p3381720`, `p3382208`, `p3468322`, `p3468325` (недогрузка).
*   **Обоснование:** Данные показывают работу насосов за пределами оптимальной зоны (BEP), что ведет к перерасходу энергии и риску повреждения оборудования.
*   **Что и как проверить:**
    1.  Проверить наличие задвижек на входе насосов `p3467585`, `p3467665` и `p3381996`, `p3381720`, `p3382208`.
    2.  Проверить соответствие производительности насосов текущей нагрузке сети.
*   **Последовательность действий:**
    1.  В модели симулировать снятие дросселирования (удаление сопротивления на входе) для насосов в зоне недогрузки.
    2.  Проверить влияние этого на общий баланс и нагрузку на другие насосы.
    3.  Для насосов в зоне перегрузки проверить возможность уменьшения их подачи (например, за счет открытия задвижек на входе или отключения в пиковые часы).
*   **Критерий проверки:** Снижение `q_to_nominal_ratio` в зоне перегрузки до < 1,15 и в зоне недогрузки до > 0,85.
*   **Ограничения:** Не устанавливать новые насосы. Не менять частотное управление без данных о возможности регулирования.
*   **Риски:** Снятие дросселирования может увеличить расход воды в сети и потребовать коррекции режима работы других насосов, что может привести к новым зонам перегрузки.

### Мероприятие 3: Устранение застоя и реверса потока
*   **Цель:** Снизить количество труб с застоем (< 0,3 м/с) и реверсом потока.
*   **Объекты:** Топовые трубы застоя: `e3447532`, `e3450285`, `e3447491`, `e3426614`; Топовые трубы реверса: `e3448407`, `e3465763`, `e3461549`.
*   **Обоснование:** Застой и реверс ведут к биологическому загрязнению и нестабильной работе сети.
*   **Что и как проверить:**
    1.  Проверить наличие потребителей на концах труб застоя.
    2.  Проверить фактическое положение задвижек на входе и выходе из этих труб.
    3.  Проверить график потребления воды в часы застоя (ночь).
*   **Последовательность действий:**
    1.  В модели установить минимальный расход (например, 0,1 м³/ч) на концах труб застоя, если там есть потребители, чтобы предотвратить полный застой.
    2.  Для труб с реверсом проверить наличие источников напорного давления, действующих в обратном направлении.
    3.  Рассмотреть возможность подключения этих труб к основной сети с постоянным потоком.
*   **Критерий проверки:** Снижение количества труб с скоростью < 0,3 м/с и реверсом.
*   **Ограничения:** Не менять диаметр труб. Не устанавливать дополнительные источники без подтверждения необходимости.
*   **Риски:** Внедрение минимального расхода может увеличить расход воды в ночное время.

### Мероприятие 4: Повышение качества данных и проверка шероховатости
*   **Цель:** Уточнить данные о состоянии труб и шероховатости для корректного моделирования.
*   **Объекты:** Топовые трубы с низкой шероховатостью: `e3444698`, `e3451591`, `e3444784`, `e3466703`.
*   **Обоснование:** Низкие баллы шероховатости (< 60) указывают на сильное зарастание, что влияет на потери напора и скорости потока.
*   **Что и как проверить:**
    1.  Проверить наличие данных о фактическом внутреннем диаметре труб в паспортах.
    2.  Измерить скорость потока и потери напора в реальных условиях.
*   **Последовательность действий:**
    1.  В модели скорректировать коэффициент шероховатости (C или n) для труб с баллами < 60, основываясь на фактическом внутреннем диаметре.
    2.  Пересчитать потери напора и скорости.
*   **Критерий проверки:** Соответствие скоростей и потерь напора реальным измерениям.
*   **Ограничения:** Не назначать замену труб без подтверждения необходимости.
*   **Риски:** Неправильная корректировка шероховатости может привести к завышению потерь и снижению напора в других узлах.


⚡ Генерация завершена: 2692 токенов за 477.4 сек (5.6 ток/сек)
Markdown и исходный ответ сохранены: reports\EPANET_AI_Report_модель семей.md
Геометрия: совпало узлов 20723 из 20723; проверьте соответствие моделей.
📊 [3/3] Готов интерактивный HTML-отчёт с графиками Кривой QH: E:\Code\EPANET_AI\reports\EPANET_AI_Report_модель семей.html
🏁 Полное время цикла «Анализ ИИ»: 479.64 сек!
